In [ ]:
# ===============================================================================
# CELL 1: INSTALLATIONS
# ===============================================================================
!pip install -q -U pandas scikit-learn requests nltk conceptnet-lite

In [ ]:
# ==============================================================================
# CÉLULA 2: IMPORTS
# ==============================================================================
import pandas as pd
from sklearn.model_selection import train_test_split
import nltk
import os
import time
import conceptnet_lite

print("Imports concluídos.")

In [ ]:
from conceptnet_lite import Label

In [ ]:
nltk.download('punkt') # Embora o erro seja 'punkt_tab', a biblioteca NLTK moderna usa 'punkt'
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')

In [ ]:
# ==============================================================================
# CÉLULA 3: DOWNLOAD DE RECURSOS DO NLTK
# ==============================================================================
nltk.download('punkt', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)
nltk.download('universal_tagset', quiet=True)

print("Recursos do NLTK baixados.")

In [ ]:
# ==============================================================================
# CÉLULA 4 (VERSÃO CORRETA E SIMPLIFICADA): CONEXÃO AUTOMÁTICA
# ==============================================================================
import conceptnet_lite
import os

# O nome do arquivo que o conceptnet_lite irá procurar ou criar no diretório /content/.
db_path = 'conceptnet.db'

conceptnet_lite.connect(db_path)

print(f"Conectando ao banco de dados local do ConceptNet ('{db_path}')...")

# Esta única linha de código cuida de tudo:
# 1. Procura por 'conceptnet.db'.
# 2. Se não encontrar, baixa o arquivo compactado (~1.85 GB).
# 3. Extrai o arquivo (isso pode demorar alguns minutos na primeira vez).
# 4. Conecta-se ao banco de dados.
# Nas execuções seguintes, ele encontrará o arquivo e se conectará instantaneamente.
conceptnet_lite.connect(db_path)

print("Conexão com ConceptNet local estabelecida com sucesso!")

In [ ]:
# ==============================================================================
# CÉLULA DE INICIALIZAÇÃO DO CONCEPTNET LITE
# ==============================================================================
import os
import conceptnet_lite

# Nome do arquivo do banco de dados
db_filename = '/content/drive/My Drive/.../conceptnet.db'

print("--- Inicializando banco de dados do ConceptNet Lite ---")

# 1. Conecta-se ao banco de dados
# Se o arquivo não existir, um novo será criado
conceptnet_lite.connect(db_filename)

# 2. Inicializa o banco de dados se ainda não tiver sido populado
# Esta é a etapa crucial que cria as tabelas e baixa os dados.
# Pode demorar alguns minutos na primeira execução.
if not os.path.exists(db_filename) or conceptnet_lite.Label.select().count() == 0:
    print(f"Banco de dados '{db_filename}' não encontrado ou vazio. Populando...")
    try:
        conceptnet_lite.DB.init()
        print("Banco de dados populado com sucesso!")
    except Exception as e:
        print(f"Erro ao popular o banco de dados: {e}")
else:
    print("Banco de dados já existe e está populado. Conectando...")

print("--- Inicialização concluída. ---")

In [ ]:
# Instala a biblioteca gdown para baixar arquivos do Google Drive
# Se já estiver instalada, esta linha não fará nada
!pip install gdown

# O ID do seu arquivo que você copiou
file_id = '1oBPesfNkZ64-9Nk2s-KukAUKnzOemMDQ'

# O nome que o arquivo terá no ambiente do Colab
output_filename = 'claim_stance_dataset_v1.csv'

# Baixa o arquivo do Google Drive
import gdown
gdown.download(f'https://drive.google.com/uc?id={file_id}', output_filename, quiet=False)

# Agora você pode ler o arquivo normalmente, pois ele está no diretório local do Colab
import pandas as pd
df = pd.read_csv(output_filename)

print("DataFrame carregado com sucesso!")
print(df.head())

In [ ]:
# ==============================================================================
# CÉLULA 5: PROCESSAMENTO DE DADOS (Como no original)
# ==============================================================================

if 'COLAB_GPU' in os.environ:
    from google.colab import drive
    print("Montando Google Drive...")
    drive.mount('/content/drive')
    df = pd.read_csv('/content/drive/MyDrive/.../claim_stance_dataset_v1.csv')
else:
    df = pd.read_csv('/content/drive/MyDrive/.../claim_stance_dataset_v1.csv')


df['claims.stance'] = df['claims.stance'].map({'PRO':'SUPPORT','CON':'ATTACK'})
df['question'] = (
    "The argument \\\"" + df['claims.claimCorrectedText'] + "\\\" "
    "SUPPORT or ATTACK the belief \\\"" + df['topicText'] + "\\\"?"
)
train_full = df[df['split']=='train']
train_df, valid_df = train_test_split(train_full, test_size=0.1, random_state=42)
test_df  = df[df['split']=='test'].copy() # Usamos .copy() para evitar warnings

print("Processamento de dados concluído.")

In [ ]:
import re

In [ ]:
# ==============================================================================
# CÉLULA 6: FUNÇÕES ADAPTADAS PARA CONCEPTNET LOCAL
# ==============================================================================
import nltk
from peewee import OperationalError
from conceptnet_lite import Label, edges_between

def extract_key_terms(text):
    """
    Extrai termos-chave, incluindo frases de substantivos (chunks), para capturar conceitos mais complexos.
    """
    tokens = nltk.word_tokenize(text.lower())
    tagged_words = nltk.pos_tag(tokens, tagset='universal')
    key_terms = []

    # Extrai termos de palavra única
    single_words = [word for word, tag in tagged_words if tag in ('NOUN', 'VERB', 'ADJ')]
    key_terms.extend(list(set(single_words)))

    # Adiciona termos compostos (bigramas) que podem ser relevantes
    bigrams = nltk.bigrams(tokens)
    for bigram in bigrams:
        if nltk.pos_tag([bigram[0]], tagset='universal')[0][1] in ('ADJ', 'NOUN') and \
           nltk.pos_tag([bigram[1]], tagset='universal')[0][1] in ('NOUN'):
            key_terms.append(' '.join(bigram))

    return list(set(key_terms))

def extract_entities_from_argument_and_topic(question_prompt):
    try:
        parts = question_prompt.split('SUPPORT or ATTACK the belief')
        argument_text = parts[0].replace('The argument', '').strip().strip('"')
        topic_text = parts[1].strip().strip('"?')

        argument_entities = extract_key_terms(argument_text)
        topic_entities = extract_key_terms(topic_text)

        return argument_entities, topic_entities
    except Exception as e:
        print(f"Erro ao parsear o prompt: {e}")
        return [], []

def find_paths_between_entities_local(argument_entities, topic_entities):
    paths = []

    # Verifique se o banco de dados do ConceptNet Lite está disponível
    try:
        Label.table_exists()
    except Exception:
        return 'Nenhum caminho direto encontrado entre os conceitos do argumento e do tópico.'

    valid_arg_entities = [entity for entity in argument_entities if Label.get_or_none(text=entity)]
    valid_topic_entities = [entity for entity in topic_entities if Label.get_or_none(text=entity)]

    # Para capturar caminhos de volta para o próprio conceito (como 'video' -> 'video')
    all_entities = list(set(valid_arg_entities + valid_topic_entities))

    for arg_entity in all_entities:
        for topic_entity in all_entities:
            try:
                edges_found = edges_between(arg_entity, topic_entity, two_way=True)
                for edge in edges_found:
                    start_label = edge.start.text
                    end_label = edge.end.text
                    rel_label = edge.relation.name
                    paths.append(f"'{start_label}' -> ({rel_label}) -> '{end_label}'")
            except OperationalError:
                continue

    if paths:
        # Use um conjunto para evitar caminhos duplicados
        return ' '.join(sorted(list(set(paths))))
    else:
        return 'Nenhum caminho direto encontrado entre os conceitos do argumento e do tópico.'

def create_prompt_with_paths_local(question_prompt):
    argument_entities, topic_entities = extract_entities_from_argument_and_topic(question_prompt)

    conceptnet_context = find_paths_between_entities_local(argument_entities, topic_entities)

    system_instruction = ("Your task is to analyze an argument and a belief, using the provided 'Knowledge Graph Context' "
                          "which shows direct paths between concepts from the argument and the belief. "
                          "Based on this, decide if the argument attacks or supports the belief. "
                          "Your answer must be ONLY the word 'ATTACK' or 'SUPPORT'.")

    full_prompt = (f"{system_instruction}\\n\\n"
                   f"Knowledge Graph Context: {conceptnet_context}\\n\\n"
                   f"Question: {question_prompt}")

    return full_prompt

print("Funções de criação de prompt definidas.")

In [ ]:
import nltk
from nltk.corpus import stopwords
import requests

# Baixa a lista de stopwords do NLTK, se ainda não estiver instalada
nltk.download('stopwords')

In [ ]:
from requests.exceptions import RequestException, JSONDecodeError

In [ ]:
# ==============================================================================
# CÉLULA 6: FUNÇÕES ADAPTADAS PARA CONCEPTNET ONLINE
# ==============================================================================
import nltk
import requests
from requests.exceptions import RequestException, JSONDecodeError
from nltk.corpus import stopwords

# Baixa a lista de stopwords do NLTK, se ainda não estiver instalada
nltk.download('stopwords')

def extract_key_terms(text):
    """
    Extrai termos-chave, incluindo frases de substantivos (chunks), para capturar conceitos mais complexos,
    utilizando a lista de stopwords do NLTK para filtragem automática.
    """
    tokens = nltk.word_tokenize(text.lower())

    english_stopwords = set(stopwords.words('english'))

    tagged_words = nltk.pos_tag(tokens, tagset='universal')
    key_terms = []

    single_words = [word for word, tag in tagged_words if tag in ('NOUN', 'VERB', 'ADJ') and word not in english_stopwords]
    key_terms.extend(list(set(single_words)))

    bigrams = nltk.bigrams(tokens)
    for bigram in bigrams:
        if bigram[0] not in english_stopwords and bigram[1] not in english_stopwords:
            if nltk.pos_tag([bigram[0]], tagset='universal')[0][1] in ('ADJ', 'NOUN') and \
               nltk.pos_tag([bigram[1]], tagset='universal')[0][1] in ('NOUN'):
                key_terms.append(' '.join(bigram))

    return list(set(key_terms))

def extract_entities_from_argument_and_topic(question_prompt):
    try:
        parts = question_prompt.split('SUPPORT or ATTACK the belief')
        argument_text = parts[0].replace('The argument', '').strip().strip('"')
        topic_text = parts[1].strip().strip('"?')

        argument_entities = extract_key_terms(argument_text)
        topic_entities = extract_key_terms(topic_text)

        return argument_entities, topic_entities
    except Exception as e:
        print(f"Erro ao parsear o prompt: {e}")
        return [], []

def find_paths_between_entities_online(argument_entities, topic_entities):
    """
    Busca por caminhos entre entidades usando a API online do ConceptNet.
    Exclui as relações 'AtLocation' e 'EtymologicallyRelated'.
    """
    paths = []

    # Relações a serem excluídas
    excluded_relations = {'AtLocation', 'EtymologicallyRelated'}

    # Combina todas as entidades para a busca
    all_entities = list(set(argument_entities + topic_entities))

    for arg_entity in all_entities:
        for topic_entity in all_entities:
            if arg_entity == topic_entity:
                continue

            print(f"🔍 Buscando caminhos entre '{arg_entity}' e '{topic_entity}'...")

            response = None

            try:
                url = f"http://api.conceptnet.io/query?start=/c/en/{arg_entity.replace(' ', '_')}&end=/c/en/{topic_entity.replace(' ', '_')}"

                response = requests.get(url)
                response.raise_for_status()

                data = response.json()

                for edge in data.get('edges', []):
                    start_label = edge['start']['label']
                    end_label = edge['end']['label']
                    rel_label = edge['rel']['label']

                    # === NOVA LÓGICA: FILTRA RELAÇÕES INDESEJADAS ===
                    if rel_label not in excluded_relations and start_label != end_label:
                        paths.append(f"'{start_label}' -> ({rel_label}) -> '{end_label}'")

            except (RequestException, JSONDecodeError) as e:
                print(f"Erro na requisição para a API do ConceptNet: {e}")
                continue

    if paths:
        return ' '.join(sorted(list(set(paths))))
    else:
        return 'Nenhum caminho direto encontrado entre os conceitos do argumento e do tópico.'

def create_prompt_with_paths_online(question_prompt):
    argument_entities, topic_entities = extract_entities_from_argument_and_topic(question_prompt)

    # Nova lógica para excluir de argument_entities o que já está em topic_entities
    argument_entities = list(set(argument_entities) - set(topic_entities))

    conceptnet_context = find_paths_between_entities_online(argument_entities, topic_entities)

    system_instruction = ("Your task is to analyze an argument and a belief, using the provided 'Knowledge Graph Context' "
                          "which shows direct paths between concepts from the argument and the belief. "
                          "Based on this, decide if the argument attacks or supports the belief. "
                          "Your answer must be ONLY the word 'ATTACK' or 'SUPPORT'.")

    full_prompt = (f"{system_instruction}\\n\\n"
                   f"Knowledge Graph Context: {conceptnet_context}\\n\\n"
                   f"Question: {question_prompt}")

    return full_prompt

In [ ]:
# ==============================================================================
# CÉLULA 6: FUNÇÕES ADAPTADAS PARA CONCEPTNET ONLINE (ASSÍNCRONAS)
# ==============================================================================
import nltk
import requests
import asyncio
import aiohttp
from requests.exceptions import RequestException, JSONDecodeError
from nltk.corpus import stopwords
import time

# Baixa a lista de stopwords do NLTK, se ainda não estiver instalada
nltk.download('stopwords')

# Relações a serem excluídas
excluded_relations = {'AtLocation', 'EtymologicallyRelated'}

def extract_key_terms(text):
    """
    Extrai termos-chave, incluindo frases de substantivos (chunks), para capturar conceitos mais complexos,
    utilizando a lista de stopwords do NLTK para filtragem automática.
    """
    tokens = nltk.word_tokenize(text.lower())

    english_stopwords = set(stopwords.words('english'))

    tagged_words = nltk.pos_tag(tokens, tagset='universal')
    key_terms = []

    single_words = [word for word, tag in tagged_words if tag in ('NOUN', 'VERB', 'ADJ') and word not in english_stopwords]
    key_terms.extend(list(set(single_words)))

    bigrams = nltk.bigrams(tokens)
    for bigram in bigrams:
        if bigram[0] not in english_stopwords and bigram[1] not in english_stopwords:
            if nltk.pos_tag([bigram[0]], tagset='universal')[0][1] in ('ADJ', 'NOUN') and \
               nltk.pos_tag([bigram[1]], tagset='universal')[0][1] in ('NOUN'):
                key_terms.append(' '.join(bigram))

    return list(set(key_terms))

def extract_entities_from_argument_and_topic(question_prompt):
    try:
        parts = question_prompt.split('SUPPORT or ATTACK the belief')
        argument_text = parts[0].replace('The argument', '').strip().strip('"')
        topic_text = parts[1].strip().strip('"?')

        argument_entities = extract_key_terms(argument_text)
        topic_entities = extract_key_terms(topic_text)

        return argument_entities, topic_entities
    except Exception as e:
        print(f"Erro ao parsear o prompt: {e}")
        return [], []

async def fetch_paths(session, arg_entity, topic_entity):
    """
    Função assíncrona para buscar caminhos entre duas entidades.
    """
    if arg_entity == topic_entity:
        return []

    url = f"http://api.conceptnet.io/query?start=/c/en/{arg_entity.replace(' ', '_')}&end=/c/en/{topic_entity.replace(' ', '_')}"

    try:
        async with session.get(url) as response:
            response.raise_for_status()
            data = await response.json()

            paths = []
            for edge in data.get('edges', []):
                start_label = edge['start']['label']
                end_label = edge['end']['label']
                rel_label = edge['rel']['label']

                if rel_label not in excluded_relations and start_label != end_label:
                    paths.append(f"'{start_label}' -> ({rel_label}) -> '{end_label}'")
            return paths

    except (aiohttp.ClientError, asyncio.TimeoutError, JSONDecodeError) as e:
        print(f"Erro na requisição para a API do ConceptNet entre '{arg_entity}' e '{topic_entity}': {e}")
        return []

async def find_paths_between_entities_online_async(argument_entities, topic_entities):
    """
    Busca por caminhos entre entidades usando a API online do ConceptNet
    de forma assíncrona para maior velocidade.
    """
    paths_list = []

    all_entities = list(set(argument_entities + topic_entities))

    async with aiohttp.ClientSession() as session:
        tasks = [asyncio.ensure_future(fetch_paths(session, arg_entity, topic_entity))
                 for arg_entity in all_entities for topic_entity in all_entities]

        results = await asyncio.gather(*tasks)

    for result_paths in results:
        paths_list.extend(result_paths)

    if paths_list:
        return ' '.join(sorted(list(set(paths_list))))
    else:
        return 'Nenhum caminho direto encontrado entre os conceitos do argumento e do tópico.'

async def create_prompt_with_paths_online(question_prompt):
    """
    Função assíncrona para criar o prompt completo.
    """
    argument_entities, topic_entities = extract_entities_from_argument_and_topic(question_prompt)

    argument_entities = list(set(argument_entities) - set(topic_entities))

    conceptnet_context = await find_paths_between_entities_online_async(argument_entities, topic_entities)

    system_instruction = ("Your task is to analyze an argument and a belief, using the provided 'Knowledge Graph Context' "
                          "which shows direct paths between concepts from the argument and the belief. "
                          "Based on this, decide if the argument attacks or supports the belief. "
                          "Your answer must be ONLY the word 'ATTACK' or 'SUPPORT'.")

    full_prompt = (f"{system_instruction}\\n\\n"
                   f"Knowledge Graph Context: {conceptnet_context}\\n\\n"
                   f"Question: {question_prompt}")

    return full_prompt

# ==============================================================================
# CÉLULA 6: FUNÇÕES ADAPTADAS PARA CONCEPTNET ONLINE (ASSÍNCRONAS COM LIMITADOR)
# ==============================================================================
import nltk
import requests
import asyncio
import aiohttp
from requests.exceptions import RequestException, JSONDecodeError
from nltk.corpus import stopwords
import time
from collections import deque

# Baixa a lista de stopwords do NLTK, se ainda não estiver instalada
nltk.download('stopwords')

# Relações a serem excluídas
excluded_relations = {'AtLocation', 'EtymologicallyRelated'}

class RateLimiter:
    """
    Controla a taxa de requisições para a API.
    A API do ConceptNet permite ~1 requisição/segundo em média (120/min).
    Este limitador garantirá que a média seja mantida.
    """
    def __init__(self, max_requests, period):
        self.max_requests = max_requests
        self.period = period
        self.timestamps = deque()

    async def wait(self):
        now = time.time()
        while self.timestamps and self.timestamps[0] <= now - self.period:
            self.timestamps.popleft()

        if len(self.timestamps) >= self.max_requests:
            sleep_time = self.timestamps[0] - (now - self.period) + 0.01
            print(f"  - Atingiu o limite de taxa. Esperando {sleep_time:.2f}s...")
            await asyncio.sleep(sleep_time)

        self.timestamps.append(time.time())

def extract_key_terms(text):
    """
    Extrai termos-chave, incluindo frases de substantivos (chunks), para capturar conceitos mais complexos,
    utilizando a lista de stopwords do NLTK para filtragem automática.
    """
    tokens = nltk.word_tokenize(text.lower())

    english_stopwords = set(stopwords.words('english'))

    tagged_words = nltk.pos_tag(tokens, tagset='universal')
    key_terms = []

    single_words = [word for word, tag in tagged_words if tag in ('NOUN', 'VERB', 'ADJ') and word not in english_stopwords]
    key_terms.extend(list(set(single_words)))

    bigrams = nltk.bigrams(tokens)
    for bigram in bigrams:
        if bigram[0] not in english_stopwords and bigram[1] not in english_stopwords:
            if nltk.pos_tag([bigram[0]], tagset='universal')[0][1] in ('ADJ', 'NOUN') and \
               nltk.pos_tag([bigram[1]], tagset='universal')[0][1] in ('NOUN'):
                key_terms.append(' '.join(bigram))

    return list(set(key_terms))

def extract_entities_from_argument_and_topic(question_prompt):
    try:
        parts = question_prompt.split('SUPPORT or ATTACK the belief')
        argument_text = parts[0].replace('The argument', '').strip().strip('"')
        topic_text = parts[1].strip().strip('"?')

        argument_entities = extract_key_terms(argument_text)
        topic_entities = extract_key_terms(topic_text)

        return argument_entities, topic_entities
    except Exception as e:
        print(f"Erro ao parsear o prompt: {e}")
        return [], []

async def fetch_paths(session, semaphore, rate_limiter, arg_entity, topic_entity):
    """
    Função assíncrona para buscar caminhos entre duas entidades, usando semáforo e limitador de taxa.
    """
    if arg_entity == topic_entity:
        return []

    async with semaphore:
        await rate_limiter.wait()
        url = f"http://api.conceptnet.io/query?start=/c/en/{arg_entity.replace(' ', '_')}&end=/c/en/{topic_entity.replace(' ', '_')}"

        try:
            async with session.get(url) as response:
                response.raise_for_status()
                data = await response.json()

                paths = []
                for edge in data.get('edges', []):
                    start_label = edge['start']['label']
                    end_label = edge['end']['label']
                    rel_label = edge['rel']['label']

                    if rel_label not in excluded_relations and start_label != end_label:
                        paths.append(f"'{start_label}' -> ({rel_label}) -> '{end_label}'")
                return paths

        except (aiohttp.ClientError, asyncio.TimeoutError, JSONDecodeError) as e:
            print(f"Erro na requisição para a API do ConceptNet entre '{arg_entity}' e '{topic_entity}': {e}")
            return []

async def find_paths_between_entities_online_async(argument_entities, topic_entities):
    """
    Busca por caminhos entre entidades usando a API online do ConceptNet
    de forma assíncrona para maior velocidade.
    """
    paths_list = []

    all_entities = list(set(argument_entities + topic_entities))

    semaphore = asyncio.Semaphore(5)
    rate_limiter = RateLimiter(max_requests=120, period=60)

    tasks = [asyncio.ensure_future(fetch_paths(session, semaphore, rate_limiter, arg_entity, topic_entity))
             for arg_entity in all_entities for topic_entity in all_entities]

    results = await asyncio.gather(*tasks)

    for result_paths in results:
        paths_list.extend(result_paths)

    if paths_list:
        return ' '.join(sorted(list(set(paths_list))))
    else:
        return 'Nenhum caminho direto encontrado entre os conceitos do argumento e do tópico.'

async def create_prompt_with_paths_online(question_prompt):
    """
    Função assíncrona para criar o prompt completo.
    """
    argument_entities, topic_entities = extract_entities_from_argument_and_topic(question_prompt)

    argument_entities = list(set(argument_entities) - set(topic_entities))

    conceptnet_context = await find_paths_between_entities_online_async(argument_entities, topic_entities)

    system_instruction = ("Your task is to analyze an argument and a belief, using the provided 'Knowledge Graph Context' "
                          "which shows direct paths between concepts from the argument and the belief. "
                          "Based on this, decide if the argument attacks or supports the belief. "
                          "Your answer must be ONLY the word 'ATTACK' or 'SUPPORT'.")

    full_prompt = (f"{system_instruction}\\n\\n"
                   f"Knowledge Graph Context: {conceptnet_context}\\n\\n"
                   f"Question: {question_prompt}")

    return full_prompt

# ==============================================================================
# CÉLULA 7: TESTE DE CASO ÚNICO ONLINE
# ==============================================================================
print("--- Executando teste de caso único para medir o tempo de criação do prompt ---")

single_prompt_text = (
    'The argument "Exposure to violent video games causes at least a temporary increase in aggression and this exposure correlates with aggression in the real world" SUPPORT or ATTACK the belief "This house believes that the sale of violent video games to minors should be banned"?'
)

async def main():
    start_time = time.time()
    prompt_gerado = await create_prompt_with_paths_online(single_prompt_text)
    end_time = time.time()

    print(f"\n⏱️ Tempo de execução para gerar o prompt: {end_time - start_time:.4f} segundos")
    print("\n--- Prompt Gerado ---")
    print(prompt_gerado)

# Chamada principal para executar a função assíncrona
# Em um notebook Colab, basta usar 'await' na chamada da função async
await main()

In [ ]:
from conceptnet_lite import Label

In [ ]:
from peewee import OperationalError

In [ ]:
# ==============================================================================
# CÉLULA 7: TESTE DE CASO ÚNICO
# ==============================================================================
print("--- Executando teste de caso único para medir o tempo de criação do prompt ---")

import time

print("--- Executando teste de caso único para medir o tempo de criação do prompt ---")

# Corrigido o uso de aspas
'''
single_prompt_text = (
    'The argument "If, for any reason, the single child is unable to care for their older adult relatives, '
    'the oldest generations would face a lack of resources and necessities" SUPPORT or ATTACK the belief '
    '"This house supports the one-child policy of the republic of China"?'
)
'''

single_prompt_text = (
    'The argument "Exposure to violent video games causes at least a temporary increase in aggression and this exposure correlates with aggression in the real world" SUPPORT or ATTACK the belief "This house believes that the sale of violent video games to minors should be banned"?'
)

start_time = time.time()
prompt_gerado = create_prompt_with_paths_local(single_prompt_text)
end_time = time.time()

print(f"\n⏱️ Tempo de execução para gerar o prompt: {end_time - start_time:.4f} segundos")
print("\n--- Prompt Gerado ---")
print(prompt_gerado)

In [ ]:
# ==============================================================================
# CÉLULA 7: TESTE DE CASO ÚNICO ONLINE
# ==============================================================================
print("--- Executando teste de caso único para medir o tempo de criação do prompt ---")

import time

print("--- Executando teste de caso único para medir o tempo de criação do prompt ---")

# Corrigido o uso de aspas
'''
single_prompt_text = (
    'The argument "If, for any reason, the single child is unable to care for their older adult relatives, '
    'the oldest generations would face a lack of resources and necessities" SUPPORT or ATTACK the belief '
    '"This house supports the one-child policy of the republic of China"?'
)
'''

single_prompt_text = (
    'The argument "Exposure to violent video games causes at least a temporary increase in aggression and this exposure correlates with aggression in the real world" SUPPORT or ATTACK the belief "This house believes that the sale of violent video games to minors should be banned"?'
)

start_time = time.time()
prompt_gerado = create_prompt_with_paths_online(single_prompt_text)
end_time = time.time()

print(f"\n⏱️ Tempo de execução para gerar o prompt: {end_time - start_time:.4f} segundos")
print("\n--- Prompt Gerado ---")
print(prompt_gerado)

In [ ]:
# ==============================================================================
# CÉLULA 8: PROCESSAMENTO EM LOTE COM CHECKPOINTING
# ==============================================================================
prompts_filename = '/content/drive/My Drive/.../prompts_IBM.csv'
processed_prompts = set()

# Verifica se o arquivo já existe e carrega os prompts já processados
if os.path.exists(prompts_filename):
    print(f"Arquivo '{prompts_filename}' encontrado. Carregando prompts já processados...")
    df_existente = pd.read_csv(prompts_filename)
    processed_prompts = set(df_existente['prompt_question'])
    print(f"{len(processed_prompts)} prompts já haviam sido processados.")
else:
    print(f"Arquivo '{prompts_filename}' não encontrado. Um novo será criado.")

# Filtra o dataframe para processar apenas os prompts que ainda não estão no arquivo
prompts_a_processar = test_df[~test_df['question'].isin(processed_prompts)]

if prompts_a_processar.empty:
    print("Todos os prompts já foram processados. Nenhuma ação necessária.")
else:
    print(f"Iniciando o processamento de {len(prompts_a_processar)} novos prompts...")

    # Itera sobre os prompts restantes
    for index, row in prompts_a_processar.iterrows():
        start_time_prompt = time.time()

        question = row['question']
        right_answer = row['claims.stance']

        # Cria o prompt enriquecido
        full_prompt = create_prompt_with_paths_local(question)

        # Prepara os dados para salvar
        novo_prompt_df = pd.DataFrame([{
            'prompt_question': question, # A pergunta original
            'full_prompt_text': full_prompt, # O prompt completo com contexto
            'right_answer': right_answer
        }])

        # Anexa ao arquivo CSV
        # O cabeçalho só é escrito se o arquivo não existir
        header = not os.path.exists(prompts_filename)
        novo_prompt_df.to_csv(prompts_filename, mode='a', header=header, index=False)

        end_time_prompt = time.time()
        print(f"Prompt {index} processado e salvo em {end_time_prompt - start_time_prompt:.2f}s.")

print("Processamento em lote concluído.")

In [ ]:
import sys
import subprocess
import os

# --- Bloco de instalação de bibliotecas ---
print("Verificando e instalando bibliotecas necessárias...")
required_packages = ["aiohttp", "pandas", "nltk", "nest_asyncio"]
for package in required_packages:
    try:
        __import__(package)
        print(f"Biblioteca '{package}' já instalada.")
    except ImportError:
        print(f"Biblioteca '{package}' não encontrada. Instalando...")
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", package])
            print(f"Biblioteca '{package}' instalada com sucesso.")
        except subprocess.CalledProcessError as e:
            print(f"Erro ao instalar a biblioteca '{package}': {e}")
            sys.exit(1)

# Recarrega as bibliotecas após a instalação, se necessário
import aiohttp
import pandas as pd
import nltk
import nest_asyncio

# Aplica o patch para permitir loops de eventos aninhados, resolvendo o RuntimeError
nest_asyncio.apply()

# Baixa a lista de stopwords do NLTK, se ainda não estiver instalada
try:
    nltk.data.find('corpora/stopwords')
    print("NLTK stopwords já baixadas.")
except nltk.downloader.DownloadError:
    print("Baixando NLTK stopwords...")
    nltk.download('stopwords')

import asyncio
from requests.exceptions import RequestException, JSONDecodeError
from nltk.corpus import stopwords
import time
import random

# Placeholder para o seu DataFrame, caso ele não tenha sido carregado
try:
    # Tenta usar o DataFrame do ambiente do usuário
    test_df
except NameError:
    print("\nAviso: O DataFrame 'test_df' não foi encontrado. Criando um DataFrame vazio para demonstração.")
    # Exemplo de como você poderia carregar seu DataFrame, se necessário
    # test_df = pd.read_csv('caminho_para_o_seu_arquivo_ibm.csv')
    test_df = pd.DataFrame(columns=['question', 'claims.stance'])
else:
    print("DataFrame 'test_df' encontrado no ambiente.")


# ==============================================================================
# CÉLULA 6: FUNÇÕES ADAPTADAS PARA CONCEPTNET ONLINE (ASSÍNCRONAS COM LIMITADOR)
# ==============================================================================
# Relações a serem excluídas
excluded_relations = {'AtLocation', 'EtymologicallyRelated'}

def extract_key_terms(text):
    """
    Extrai termos-chave, incluindo frases de substantivos (chunks), para capturar conceitos mais complexos,
    utilizando a lista de stopwords do NLTK para filtragem automática.
    """
    tokens = nltk.word_tokenize(text.lower())

    english_stopwords = set(stopwords.words('english'))

    tagged_words = nltk.pos_tag(tokens, tagset='universal')
    key_terms = []

    single_words = [word for word, tag in tagged_words if tag in ('NOUN', 'VERB', 'ADJ') and word not in english_stopwords]
    key_terms.extend(list(set(single_words)))

    bigrams = nltk.bigrams(tokens)
    for bigram in bigrams:
        if bigram[0] not in english_stopwords and bigram[1] not in english_stopwords:
            if nltk.pos_tag([bigram[0]], tagset='universal')[0][1] in ('ADJ', 'NOUN') and \
               nltk.pos_tag([bigram[1]], tagset='universal')[0][1] in ('NOUN'):
                key_terms.append(' '.join(bigram))

    return list(set(key_terms))

def extract_entities_from_argument_and_topic(question_prompt):
    try:
        parts = question_prompt.split('SUPPORT or ATTACK the belief')
        argument_text = parts[0].replace('The argument', '').strip().strip('"')
        topic_text = parts[1].strip().strip('"?')

        argument_entities = extract_key_terms(argument_text)
        topic_entities = extract_key_terms(topic_text)

        return argument_entities, topic_entities
    except Exception as e:
        print(f"Erro ao parsear o prompt: {e}")
        return [], []

async def fetch_paths(session, semaphore, rate_limiter, arg_entity, topic_entity):
    """
    Função assíncrona para buscar caminhos entre duas entidades, usando semáforo e limitador de taxa.
    """
    if arg_entity == topic_entity:
        return []

    async with semaphore:
        await rate_limiter.wait()
        url = f"http://api.conceptnet.io/query?start=/c/en/{arg_entity.replace(' ', '_')}&end=/c/en/{topic_entity.replace(' ', '_')}"

        try:
            async with session.get(url) as response:
                response.raise_for_status()
                data = await response.json()

                paths = []
                for edge in data.get('edges', []):
                    start_label = edge['start']['label']
                    end_label = edge['end']['label']
                    rel_label = edge['rel']['label']

                    if rel_label not in excluded_relations and start_label != end_label:
                        paths.append(f"'{start_label}' -> ({rel_label}) -> '{end_label}'")
                return paths

        except (aiohttp.ClientError, asyncio.TimeoutError, JSONDecodeError) as e:
            print(f"Erro na requisição para a API do ConceptNet entre '{arg_entity}' e '{topic_entity}': {e}")
            return []

async def find_paths_between_entities_online_async(argument_entities, topic_entities):
    """
    Busca por caminhos entre entidades usando a API online do ConceptNet
    de forma assíncrona para maior velocidade.
    """
    paths_list = []

    all_entities = list(set(argument_entities + topic_entities))

    semaphore = asyncio.Semaphore(5)
    rate_limiter = RateLimiter(max_requests=120, period=60)

    tasks = [asyncio.ensure_future(fetch_paths(session, semaphore, rate_limiter, arg_entity, topic_entity))
             for arg_entity in all_entities for topic_entity in all_entities]

    results = await asyncio.gather(*tasks)

    for result_paths in results:
        paths_list.extend(result_paths)

    if paths_list:
        return ' '.join(sorted(list(set(paths_list))))
    else:
        return 'Nenhum caminho direto encontrado entre os conceitos do argumento e do tópico.'

async def create_prompt_with_paths_online(question_prompt):
    """
    Função assíncrona para criar o prompt completo.
    """
    argument_entities, topic_entities = extract_entities_from_argument_and_topic(question_prompt)

    argument_entities = list(set(argument_entities) - set(topic_entities))

    conceptnet_context = await find_paths_between_entities_online_async(argument_entities, topic_entities)

    system_instruction = ("Your task is to analyze an argument and a belief, using the provided 'Knowledge Graph Context' "
                          "which shows direct paths between concepts from the argument and the belief. "
                          "Based on this, decide if the argument attacks or supports the belief. "
                          "Your answer must be ONLY the word 'ATTACK' or 'SUPPORT'.")

    full_prompt = (f"{system_instruction}\\n\\n"
                   f"Knowledge Graph Context: {conceptnet_context}\\n\\n"
                   f"Question: {question_prompt}")

    return full_prompt

# ==============================================================================
# CÉLULA 7: TESTE DE CASO ÚNICO ONLINE
# ==============================================================================
print("--- Executando teste de caso único para medir o tempo de criação do prompt ---")

single_prompt_text = (
    'The argument "Exposure to violent video games causes at least a temporary increase in aggression and this exposure correlates with aggression in the real world" SUPPORT or ATTACK the belief "This house believes that the sale of violent video games to minors should be banned"?'
)

async def main():
    start_time = time.time()
    prompt_gerado = await create_prompt_with_paths_online(single_prompt_text)
    end_time = time.time()

    print(f"\n⏱️ Tempo de execução para gerar o prompt: {end_time - start_time:.4f} segundos")
    print("\n--- Prompt Gerado ---")
    print(prompt_gerado)

# Chamada principal para executar a função assíncrona
# Em um notebook Colab, basta usar 'await' na chamada da função async
await main()

In [ ]:
from conceptnet_lite import Label

In [ ]:
from peewee import OperationalError

In [ ]:
# ==============================================================================
# CÉLULA 7: TESTE DE CASO ÚNICO
# ==============================================================================
print("--- Executando teste de caso único para medir o tempo de criação do prompt ---")

import time

print("--- Executando teste de caso único para medir o tempo de criação do prompt ---")

# Corrigido o uso de aspas
'''
single_prompt_text = (
    'The argument "If, for any reason, the single child is unable to care for their older adult relatives, '
    'the oldest generations would face a lack of resources and necessities" SUPPORT or ATTACK the belief '
    '"This house supports the one-child policy of the republic of China"?'
)
'''

single_prompt_text = (
    'The argument "Exposure to violent video games causes at least a temporary increase in aggression and this exposure correlates with aggression in the real world" SUPPORT or ATTACK the belief "This house believes that the sale of violent video games to minors should be banned"?'
)

start_time = time.time()
prompt_gerado = create_prompt_with_paths_local(single_prompt_text)
end_time = time.time()

print(f"\n⏱️ Tempo de execução para gerar o prompt: {end_time - start_time:.4f} segundos")
print("\n--- Prompt Gerado ---")
print(prompt_gerado)

In [ ]:
# ==============================================================================
# CÉLULA 7: TESTE DE CASO ÚNICO ONLINE
# ==============================================================================
print("--- Executando teste de caso único para medir o tempo de criação do prompt ---")

import time

print("--- Executando teste de caso único para medir o tempo de criação do prompt ---")

# Corrigido o uso de aspas
'''
single_prompt_text = (
    'The argument "If, for any reason, the single child is unable to care for their older adult relatives, '
    'the oldest generations would face a lack of resources and necessities" SUPPORT or ATTACK the belief '
    '"This house supports the one-child policy of the republic of China"?'
)
'''

single_prompt_text = (
    'The argument "Exposure to violent video games causes at least a temporary increase in aggression and this exposure correlates with aggression in the real world" SUPPORT or ATTACK the belief "This house believes that the sale of violent video games to minors should be banned"?'
)

start_time = time.time()
prompt_gerado = create_prompt_with_paths_online(single_prompt_text)
end_time = time.time()

print(f"\n⏱️ Tempo de execução para gerar o prompt: {end_time - start_time:.4f} segundos")
print("\n--- Prompt Gerado ---")
print(prompt_gerado)

In [ ]:
# ==============================================================================
# CÉLULA 8: PROCESSAMENTO EM LOTE COM CHECKPOINTING
# ==============================================================================
prompts_filename = '/content/drive/My Drive/.../prompts_IBM.csv'
processed_prompts = set()

# Verifica se o arquivo já existe e carrega os prompts já processados
if os.path.exists(prompts_filename):
    print(f"Arquivo '{prompts_filename}' encontrado. Carregando prompts já processados...")
    df_existente = pd.read_csv(prompts_filename)
    processed_prompts = set(df_existente['prompt_question'])
    print(f"{len(processed_prompts)} prompts já haviam sido processados.")
else:
    print(f"Arquivo '{prompts_filename}' não encontrado. Um novo será criado.")

# Filtra o dataframe para processar apenas os prompts que ainda não estão no arquivo
prompts_a_processar = test_df[~test_df['question'].isin(processed_prompts)]

if prompts_a_processar.empty:
    print("Todos os prompts já foram processados. Nenhuma ação necessária.")
else:
    print(f"Iniciando o processamento de {len(prompts_a_processar)} novos prompts...")

    # Itera sobre os prompts restantes
    for index, row in prompts_a_processar.iterrows():
        start_time_prompt = time.time()

        question = row['question']
        right_answer = row['claims.stance']

        # Cria o prompt enriquecido
        full_prompt = create_prompt_with_paths_local(question)

        # Prepara os dados para salvar
        novo_prompt_df = pd.DataFrame([{
            'prompt_question': question, # A pergunta original
            'full_prompt_text': full_prompt, # O prompt completo com contexto
            'right_answer': right_answer
        }])

        # Anexa ao arquivo CSV
        # O cabeçalho só é escrito se o arquivo não existir
        header = not os.path.exists(prompts_filename)
        novo_prompt_df.to_csv(prompts_filename, mode='a', header=header, index=False)

        end_time_prompt = time.time()
        print(f"Prompt {index} processado e salvo em {end_time_prompt - start_time_prompt:.2f}s.")

print("Processamento em lote concluído.")

In [ ]:
import sys
import subprocess
import os

# --- Bloco de instalação de bibliotecas ---
print("Verificando e instalando bibliotecas necessárias...")
required_packages = ["aiohttp", "pandas", "nltk", "nest_asyncio"]
for package in required_packages:
    try:
        __import__(package)
        print(f"Biblioteca '{package}' já instalada.")
    except ImportError:
        print(f"Biblioteca '{package}' não encontrada. Instalando...")
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", package])
            print(f"Biblioteca '{package}' instalada com sucesso.")
        except subprocess.CalledProcessError as e:
            print(f"Erro ao instalar a biblioteca '{package}': {e}")
            sys.exit(1)

# Recarrega as bibliotecas após a instalação, se necessário
import aiohttp
import pandas as pd
import nltk
import nest_asyncio

# Aplica o patch para permitir loops de eventos aninhados, resolvendo o RuntimeError
nest_asyncio.apply()

# Baixa a lista de stopwords do NLTK, se ainda não estiver instalada
try:
    nltk.data.find('corpora/stopwords')
    print("NLTK stopwords já baixadas.")
except nltk.downloader.DownloadError:
    print("Baixando NLTK stopwords...")
    nltk.download('stopwords')

import asyncio
from requests.exceptions import RequestException, JSONDecodeError
from nltk.corpus import stopwords
import time
import random

# Placeholder para o seu DataFrame, caso ele não tenha sido carregado
try:
    # Tenta usar o DataFrame do ambiente do usuário
    test_df
except NameError:
    print("\nAviso: O DataFrame 'test_df' não foi encontrado. Criando um DataFrame vazio para demonstração.")
    # Exemplo de como você poderia carregar seu DataFrame, se necessário
    # test_df = pd.read_csv('caminho_para_o_seu_arquivo_ibm.csv')
    test_df = pd.DataFrame(columns=['question', 'claims.stance'])
else:
    print("DataFrame 'test_df' encontrado no ambiente.")


# ==============================================================================
# CÉLULA 6: FUNÇÕES ADAPTADAS PARA CONCEPTNET ONLINE (ASSÍNCRONAS COM LIMITADOR)
# ==============================================================================
# Relações a serem excluídas
excluded_relations = {'AtLocation', 'EtymologicallyRelated'}

def extract_key_terms(text):
    """
    Extrai termos-chave, incluindo frases de substantivos (chunks), para capturar conceitos mais complexos,
    utilizando a lista de stopwords do NLTK para filtragem automática.
    """
    tokens = nltk.word_tokenize(text.lower())

    english_stopwords = set(stopwords.words('english'))

    tagged_words = nltk.pos_tag(tokens, tagset='universal')
    key_terms = []

    single_words = [word for word, tag in tagged_words if tag in ('NOUN', 'VERB', 'ADJ') and word not in english_stopwords]
    key_terms.extend(list(set(single_words)))

    bigrams = nltk.bigrams(tokens)
    for bigram in bigrams:
        if bigram[0] not in english_stopwords and bigram[1] not in english_stopwords:
            if nltk.pos_tag([bigram[0]], tagset='universal')[0][1] in ('ADJ', 'NOUN') and \
               nltk.pos_tag([bigram[1]], tagset='universal')[0][1] in ('NOUN'):
                key_terms.append(' '.join(bigram))

    return list(set(key_terms))

def extract_entities_from_argument_and_topic(question_prompt):
    try:
        parts = question_prompt.split('SUPPORT or ATTACK the belief')
        argument_text = parts[0].replace('The argument', '').strip().strip('"')
        topic_text = parts[1].strip().strip('"?')

        argument_entities = extract_key_terms(argument_text)
        topic_entities = extract_key_terms(topic_text)

        return argument_entities, topic_entities
    except Exception as e:
        print(f"Erro ao parsear o prompt: {e}")
        return [], []

async def fetch_paths(session, semaphore, rate_limiter, arg_entity, topic_entity):
    """
    Função assíncrona para buscar caminhos entre duas entidades, usando semáforo e limitador de taxa.
    """
    if arg_entity == topic_entity:
        return []

    async with semaphore:
        await rate_limiter.wait()
        url = f"http://api.conceptnet.io/query?start=/c/en/{arg_entity.replace(' ', '_')}&end=/c/en/{topic_entity.replace(' ', '_')}"

        try:
            async with session.get(url) as response:
                response.raise_for_status()
                data = await response.json()

                paths = []
                for edge in data.get('edges', []):
                    start_label = edge['start']['label']
                    end_label = edge['end']['label']
                    rel_label = edge['rel']['label']

                    if rel_label not in excluded_relations and start_label != end_label:
                        paths.append(f"'{start_label}' -> ({rel_label}) -> '{end_label}'")
                return paths

        except (aiohttp.ClientError, asyncio.TimeoutError, JSONDecodeError) as e:
            print(f"Erro na requisição para a API do ConceptNet entre '{arg_entity}' e '{topic_entity}': {e}")
            return []

async def find_paths_between_entities_online_async(argument_entities, topic_entities):
    """
    Busca por caminhos entre entidades usando a API online do ConceptNet
    de forma assíncrona para maior velocidade.
    """
    paths_list = []

    all_entities = list(set(argument_entities + topic_entities))

    semaphore = asyncio.Semaphore(5)
    rate_limiter = RateLimiter(max_requests=120, period=60)

    tasks = [asyncio.ensure_future(fetch_paths(session, semaphore, rate_limiter, arg_entity, topic_entity))
             for arg_entity in all_entities for topic_entity in all_entities]

    results = await asyncio.gather(*tasks)

    for result_paths in results:
        paths_list.extend(result_paths)

    if paths_list:
        return ' '.join(sorted(list(set(paths_list))))
    else:
        return 'Nenhum caminho direto encontrado entre os conceitos do argumento e do tópico.'

async def create_prompt_with_paths_online(question_prompt):
    """
    Função assíncrona para criar o prompt completo.
    """
    argument_entities, topic_entities = extract_entities_from_argument_and_topic(question_prompt)

    argument_entities = list(set(argument_entities) - set(topic_entities))

    conceptnet_context = await find_paths_between_entities_online_async(argument_entities, topic_entities)

    system_instruction = ("Your task is to analyze an argument and a belief, using the provided 'Knowledge Graph Context' "
                          "which shows direct paths between concepts from the argument and the belief. "
                          "Based on this, decide if the argument attacks or supports the belief. "
                          "Your answer must be ONLY the word 'ATTACK' or 'SUPPORT'.")

    full_prompt = (f"{system_instruction}\\n\\n"
                   f"Knowledge Graph Context: {conceptnet_context}\\n\\n"
                   f"Question: {question_prompt}")

    return full_prompt

# ==============================================================================
# CÉLULA 7: TESTE DE CASO ÚNICO ONLINE
# ==============================================================================
print("--- Executando teste de caso único para medir o tempo de criação do prompt ---")

single_prompt_text = (
    'The argument "Exposure to violent video games causes at least a temporary increase in aggression and this exposure correlates with aggression in the real world" SUPPORT or ATTACK the belief "This house believes that the sale of violent video games to minors should be banned"?'
)

async def main():
    start_time = time.time()
    prompt_gerado = await create_prompt_with_paths_online(single_prompt_text)
    end_time = time.time()

    print(f"\n⏱️ Tempo de execução para gerar o prompt: {end_time - start_time:.4f} segundos")
    print("\n--- Prompt Gerado ---")
    print(prompt_gerado)

# Chamada principal para executar a função assíncrona
# Em um notebook Colab, basta usar 'await' na chamada da função async
await main()

In [ ]:
from conceptnet_lite import Label

In [ ]:
from peewee import OperationalError

In [ ]:
# ==============================================================================
# CÉLULA 7: TESTE DE CASO ÚNICO
# ==============================================================================
print("--- Executando teste de caso único para medir o tempo de criação do prompt ---")

import time

print("--- Executando teste de caso único para medir o tempo de criação do prompt ---")

# Corrigido o uso de aspas
'''
single_prompt_text = (
    'The argument "If, for any reason, the single child is unable to care for their older adult relatives, '
    'the oldest generations would face a lack of resources and necessities" SUPPORT or ATTACK the belief '
    '"This house supports the one-child policy of the republic of China"?'
)
'''

single_prompt_text = (
    'The argument "Exposure to violent video games causes at least a temporary increase in aggression and this exposure correlates with aggression in the real world" SUPPORT or ATTACK the belief "This house believes that the sale of violent video games to minors should be banned"?'
)

start_time = time.time()
prompt_gerado = create_prompt_with_paths_local(single_prompt_text)
end_time = time.time()

print(f"\n⏱️ Tempo de execução para gerar o prompt: {end_time - start_time:.4f} segundos")
print("\n--- Prompt Gerado ---")
print(prompt_gerado)

In [ ]:
# ==============================================================================
# CÉLULA 7: TESTE DE CASO ÚNICO ONLINE
# ==============================================================================
print("--- Executando teste de caso único para medir o tempo de criação do prompt ---")

import time

print("--- Executando teste de caso único para medir o tempo de criação do prompt ---")

# Corrigido o uso de aspas
'''
single_prompt_text = (
    'The argument "If, for any reason, the single child is unable to care for their older adult relatives, '
    'the oldest generations would face a lack of resources and necessities" SUPPORT or ATTACK the belief '
    '"This house supports the one-child policy of the republic of China"?'
)
'''

single_prompt_text = (
    'The argument "Exposure to violent video games causes at least a temporary increase in aggression and this exposure correlates with aggression in the real world" SUPPORT or ATTACK the belief "This house believes that the sale of violent video games to minors should be banned"?'
)

start_time = time.time()
prompt_gerado = create_prompt_with_paths_online(single_prompt_text)
end_time = time.time()

print(f"\n⏱️ Tempo de execução para gerar o prompt: {end_time - start_time:.4f} segundos")
print("\n--- Prompt Gerado ---")
print(prompt_gerado)

In [ ]:
# ==============================================================================
# CÉLULA 8: PROCESSAMENTO EM LOTE COM CHECKPOINTING
# ==============================================================================
prompts_filename = '/content/drive/My Drive/.../prompts_IBM.csv'
processed_prompts = set()

# Verifica se o arquivo já existe e carrega os prompts já processados
if os.path.exists(prompts_filename):
    print(f"Arquivo '{prompts_filename}' encontrado. Carregando prompts já processados...")
    df_existente = pd.read_csv(prompts_filename)
    processed_prompts = set(df_existente['prompt_question'])
    print(f"{len(processed_prompts)} prompts já haviam sido processados.")
else:
    print(f"Arquivo '{prompts_filename}' não encontrado. Um novo será criado.")

# Filtra o dataframe para processar apenas os prompts que ainda não estão no arquivo
prompts_a_processar = test_df[~test_df['question'].isin(processed_prompts)]

if prompts_a_processar.empty:
    print("Todos os prompts já foram processados. Nenhuma ação necessária.")
else:
    print(f"Iniciando o processamento de {len(prompts_a_processar)} novos prompts...")

    # Itera sobre os prompts restantes
    for index, row in prompts_a_processar.iterrows():
        start_time_prompt = time.time()

        question = row['question']
        right_answer = row['claims.stance']

        # Cria o prompt enriquecido
        full_prompt = create_prompt_with_paths_local(question)

        # Prepara os dados para salvar
        novo_prompt_df = pd.DataFrame([{
            'prompt_question': question, # A pergunta original
            'full_prompt_text': full_prompt, # O prompt completo com contexto
            'right_answer': right_answer
        }])

        # Anexa ao arquivo CSV
        # O cabeçalho só é escrito se o arquivo não existir
        header = not os.path.exists(prompts_filename)
        novo_prompt_df.to_csv(prompts_filename, mode='a', header=header, index=False)

        end_time_prompt = time.time()
        print(f"Prompt {index} processado e salvo em {end_time_prompt - start_time_prompt:.2f}s.")

print("Processamento em lote concluído.")

In [ ]:
import sys
import subprocess
import os

# --- Bloco de instalação de bibliotecas ---
print("Verificando e instalando bibliotecas necessárias...")
required_packages = ["aiohttp", "pandas", "nltk", "nest_asyncio"]
for package in required_packages:
    try:
        __import__(package)
        print(f"Biblioteca '{package}' já instalada.")
    except ImportError:
        print(f"Biblioteca '{package}' não encontrada. Instalando...")
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", package])
            print(f"Biblioteca '{package}' instalada com sucesso.")
        except subprocess.CalledProcessError as e:
            print(f"Erro ao instalar a biblioteca '{package}': {e}")
            sys.exit(1)

# Recarrega as bibliotecas após a instalação, se necessário
import aiohttp
import pandas as pd
import nltk
import nest_asyncio

# Aplica o patch para permitir loops de eventos aninhados, resolvendo o RuntimeError
nest_asyncio.apply()

# Baixa a lista de stopwords do NLTK, se ainda não estiver instalada
try:
    nltk.data.find('corpora/stopwords')
    print("NLTK stopwords já baixadas.")
except nltk.downloader.DownloadError:
    print("Baixando NLTK stopwords...")
    nltk.download('stopwords')

import asyncio
from requests.exceptions import RequestException, JSONDecodeError
from nltk.corpus import stopwords
import time
import random

# Placeholder para o seu DataFrame, caso ele não tenha sido carregado
try:
    # Tenta usar o DataFrame do ambiente do usuário
    test_df
except NameError:
    print("\nAviso: O DataFrame 'test_df' não foi encontrado. Criando um DataFrame vazio para demonstração.")
    # Exemplo de como você poderia carregar seu DataFrame, se necessário
    # test_df = pd.read_csv('caminho_para_o_seu_arquivo_ibm.csv')
    test_df = pd.DataFrame(columns=['question', 'claims.stance'])
else:
    print("DataFrame 'test_df' encontrado no ambiente.")


# ==============================================================================
# CÉLULA 6: FUNÇÕES ADAPTADAS PARA CONCEPTNET ONLINE (ASSÍNCRONAS COM LIMITADOR)
# ==============================================================================
# Relações a serem excluídas
excluded_relations = {'AtLocation', 'EtymologicallyRelated'}

def extract_key_terms(text):
    """
    Extrai termos-chave, incluindo frases de substantivos (chunks), para capturar conceitos mais complexos,
    utilizando a lista de stopwords do NLTK para filtragem automática.
    """
    tokens = nltk.word_tokenize(text.lower())

    english_stopwords = set(stopwords.words('english'))

    tagged_words = nltk.pos_tag(tokens, tagset='universal')
    key_terms = []

    single_words = [word for word, tag in tagged_words if tag in ('NOUN', 'VERB', 'ADJ') and word not in english_stopwords]
    key_terms.extend(list(set(single_words)))

    bigrams = nltk.bigrams(tokens)
    for bigram in bigrams:
        if bigram[0] not in english_stopwords and bigram[1] not in english_stopwords:
            if nltk.pos_tag([bigram[0]], tagset='universal')[0][1] in ('ADJ', 'NOUN') and \
               nltk.pos_tag([bigram[1]], tagset='universal')[0][1] in ('NOUN'):
                key_terms.append(' '.join(bigram))

    return list(set(key_terms))

def extract_entities_from_argument_and_topic(question_prompt):
    try:
        parts = question_prompt.split('SUPPORT or ATTACK the belief')
        argument_text = parts[0].replace('The argument', '').strip().strip('"')
        topic_text = parts[1].strip().strip('"?')

        argument_entities = extract_key_terms(argument_text)
        topic_entities = extract_key_terms(topic_text)

        return argument_entities, topic_entities
    except Exception as e:
        print(f"Erro ao parsear o prompt: {e}")
        return [], []

async def fetch_paths(session, semaphore, rate_limiter, arg_entity, topic_entity):
    """
    Função assíncrona para buscar caminhos entre duas entidades, usando semáforo e limitador de taxa.
    """
    if arg_entity == topic_entity:
        return []

    async with semaphore:
        await rate_limiter.wait()
        url = f"http://api.conceptnet.io/query?start=/c/en/{arg_entity.replace(' ', '_')}&end=/c/en/{topic_entity.replace(' ', '_')}"

        try:
            async with session.get(url) as response:
                response.raise_for_status()
                data = await response.json()

                paths = []
                for edge in data.get('edges', []):
                    start_label = edge['start']['label']
                    end_label = edge['end']['label']
                    rel_label = edge['rel']['label']

                    if rel_label not in excluded_relations and start_label != end_label:
                        paths.append(f"'{start_label}' -> ({rel_label}) -> '{end_label}'")
                return paths

        except (aiohttp.ClientError, asyncio.TimeoutError, JSONDecodeError) as e:
            print(f"Erro na requisição para a API do ConceptNet entre '{arg_entity}' e '{topic_entity}': {e}")
            return []

async def find_paths_between_entities_online_async(argument_entities, topic_entities):
    """
    Busca por caminhos entre entidades usando a API online do ConceptNet
    de forma assíncrona para maior velocidade.
    """
    paths_list = []

    all_entities = list(set(argument_entities + topic_entities))

    semaphore = asyncio.Semaphore(5)
    rate_limiter = RateLimiter(max_requests=120, period=60)

    tasks = [asyncio.ensure_future(fetch_paths(session, semaphore, rate_limiter, arg_entity, topic_entity))
             for arg_entity in all_entities for topic_entity in all_entities]

    results = await asyncio.gather(*tasks)

    for result_paths in results:
        paths_list.extend(result_paths)

    if paths_list:
        return ' '.join(sorted(list(set(paths_list))))
    else:
        return 'Nenhum caminho direto encontrado entre os conceitos do argumento e do tópico.'

async def create_prompt_with_paths_online(question_prompt):
    """
    Função assíncrona para criar o prompt completo.
    """
    argument_entities, topic_entities = extract_entities_from_argument_and_topic(question_prompt)

    argument_entities = list(set(argument_entities) - set(topic_entities))

    conceptnet_context = await find_paths_between_entities_online_async(argument_entities, topic_entities)

    system_instruction = ("Your task is to analyze an argument and a belief, using the provided 'Knowledge Graph Context' "
                          "which shows direct paths between concepts from the argument and the belief. "
                          "Based on this, decide if the argument attacks or supports the belief. "
                          "Your answer must be ONLY the word 'ATTACK' or 'SUPPORT'.")

    full_prompt = (f"{system_instruction}\\n\\n"
                   f"Knowledge Graph Context: {conceptnet_context}\\n\\n"
                   f"Question: {question_prompt}")

    return full_prompt

# ==============================================================================
# CÉLULA 7: TESTE DE CASO ÚNICO ONLINE
# ==============================================================================
print("--- Executando teste de caso único para medir o tempo de criação do prompt ---")

single_prompt_text = (
    'The argument "Exposure to violent video games causes at least a temporary increase in aggression and this exposure correlates with aggression in the real world" SUPPORT or ATTACK the belief "This house believes that the sale of violent video games to minors should be banned"?'
)

async def main():
    start_time = time.time()
    prompt_gerado = await create_prompt_with_paths_online(single_prompt_text)
    end_time = time.time()

    print(f"\n⏱️ Tempo de execução para gerar o prompt: {end_time - start_time:.4f} segundos")
    print("\n--- Prompt Gerado ---")
    print(prompt_gerado)

# Chamada principal para executar a função assíncrona
# Em um notebook Colab, basta usar 'await' na chamada da função async
await main()

In [ ]:
from conceptnet_lite import Label

In [ ]:
from peewee import OperationalError

In [ ]:
# ==============================================================================
# CÉLULA 7: TESTE DE CASO ÚNICO
# ==============================================================================
print("--- Executando teste de caso único para medir o tempo de criação do prompt ---")

import time

print("--- Executando teste de caso único para medir o tempo de criação do prompt ---")

# Corrigido o uso de aspas
'''
single_prompt_text = (
    'The argument "If, for any reason, the single child is unable to care for their older adult relatives, '
    'the oldest generations would face a lack of resources and necessities" SUPPORT or ATTACK the belief '
    '"This house supports the one-child policy of the republic of China"?'
)
'''

single_prompt_text = (
    'The argument "Exposure to violent video games causes at least a temporary increase in aggression and this exposure correlates with aggression in the real world" SUPPORT or ATTACK the belief "This house believes that the sale of violent video games to minors should be banned"?'
)

start_time = time.time()
prompt_gerado = create_prompt_with_paths_local(single_prompt_text)
end_time = time.time()

print(f"\n⏱️ Tempo de execução para gerar o prompt: {end_time - start_time:.4f} segundos")
print("\n--- Prompt Gerado ---")
print(prompt_gerado)

In [ ]:
# ==============================================================================
# CÉLULA 7: TESTE DE CASO ÚNICO ONLINE
# ==============================================================================
print("--- Executando teste de caso único para medir o tempo de criação do prompt ---")

import time

print("--- Executando teste de caso único para medir o tempo de criação do prompt ---")

# Corrigido o uso de aspas
'''
single_prompt_text = (
    'The argument "If, for any reason, the single child is unable to care for their older adult relatives, '
    'the oldest generations would face a lack of resources and necessities" SUPPORT or ATTACK the belief '
    '"This house supports the one-child policy of the republic of China"?'
)
'''

single_prompt_text = (
    'The argument "Exposure to violent video games causes at least a temporary increase in aggression and this exposure correlates with aggression in the real world" SUPPORT or ATTACK the belief "This house believes that the sale of violent video games to minors should be banned"?'
)

start_time = time.time()
prompt_gerado = create_prompt_with_paths_online(single_prompt_text)
end_time = time.time()

print(f"\n⏱️ Tempo de execução para gerar o prompt: {end_time - start_time:.4f} segundos")
print("\n--- Prompt Gerado ---")
print(prompt_gerado)

In [ ]:
# ==============================================================================
# CÉLULA 8: PROCESSAMENTO EM LOTE COM CHECKPOINTING
# ==============================================================================
prompts_filename = '/content/drive/My Drive/.../prompts_IBM.csv'
processed_prompts = set()

# Verifica se o arquivo já existe e carrega os prompts já processados
if os.path.exists(prompts_filename):
    print(f"Arquivo '{prompts_filename}' encontrado. Carregando prompts já processados...")
    df_existente = pd.read_csv(prompts_filename)
    processed_prompts = set(df_existente['prompt_question'])
    print(f"{len(processed_prompts)} prompts já haviam sido processados.")
else:
    print(f"Arquivo '{prompts_filename}' não encontrado. Um novo será criado.")

# Filtra o dataframe para processar apenas os prompts que ainda não estão no arquivo
prompts_a_processar = test_df[~test_df['question'].isin(processed_prompts)]

if prompts_a_processar.empty:
    print("Todos os prompts já foram processados. Nenhuma ação necessária.")
else:
    print(f"Iniciando o processamento de {len(prompts_a_processar)} novos prompts...")

    # Itera sobre os prompts restantes
    for index, row in prompts_a_processar.iterrows():
        start_time_prompt = time.time()

        question = row['question']
        right_answer = row['claims.stance']

        # Cria o prompt enriquecido
        full_prompt = create_prompt_with_paths_local(question)

        # Prepara os dados para salvar
        novo_prompt_df = pd.DataFrame([{
            'prompt_question': question, # A pergunta original
            'full_prompt_text': full_prompt, # O prompt completo com contexto
            'right_answer': right_answer
        }])

        # Anexa ao arquivo CSV
        # O cabeçalho só é escrito se o arquivo não existir
        header = not os.path.exists(prompts_filename)
        novo_prompt_df.to_csv(prompts_filename, mode='a', header=header, index=False)

        end_time_prompt = time.time()
        print(f"Prompt {index} processado e salvo em {end_time_prompt - start_time_prompt:.2f}s.")

print("Processamento em lote concluído.")

In [ ]:
import sys
import subprocess
import os

# --- Bloco de instalação de bibliotecas ---
print("Verificando e instalando bibliotecas necessárias...")
required_packages = ["aiohttp", "pandas", "nltk", "nest_asyncio"]
for package in required_packages:
    try:
        __import__(package)
        print(f"Biblioteca '{package}' já instalada.")
    except ImportError:
        print(f"Biblioteca '{package}' não encontrada. Instalando...")
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", package])
            print(f"Biblioteca '{package}' instalada com sucesso.")
        except subprocess.CalledProcessError as e:
            print(f"Erro ao instalar a biblioteca '{package}': {e}")
            sys.exit(1)

# Recarrega as bibliotecas após a instalação, se necessário
import aiohttp
import pandas as pd
import nltk
import nest_asyncio

# Aplica o patch para permitir loops de eventos aninhados, resolvendo o RuntimeError
nest_asyncio.apply()

# Baixa a lista de stopwords do NLTK, se ainda não estiver instalada
try:
    nltk.data.find('corpora/stopwords')
    print("NLTK stopwords já baixadas.")
except nltk.downloader.DownloadError:
    print("Baixando NLTK stopwords...")
    nltk.download('stopwords')

import asyncio
from requests.exceptions import RequestException, JSONDecodeError
from nltk.corpus import stopwords
import time
import random

# Placeholder para o seu DataFrame, caso ele não tenha sido carregado
try:
    # Tenta usar o DataFrame do ambiente do usuário
    test_df
except NameError:
    print("\nAviso: O DataFrame 'test_df' não foi encontrado. Criando um DataFrame vazio para demonstração.")
    # Exemplo de como você poderia carregar seu DataFrame, se necessário
    # test_df = pd.read_csv('caminho_para_o_seu_arquivo_ibm.csv')
    test_df = pd.DataFrame(columns=['question', 'claims.stance'])
else:
    print("DataFrame 'test_df' encontrado no ambiente.")


# ==============================================================================
# CÉLULA 6: FUNÇÕES ADAPTADAS PARA CONCEPTNET ONLINE (ASSÍNCRONAS COM LIMITADOR)
# ==============================================================================
# Relações a serem excluídas
excluded_relations = {'AtLocation', 'EtymologicallyRelated'}

def extract_key_terms(text):
    """
    Extrai termos-chave, incluindo frases de substantivos (chunks), para capturar conceitos mais complexos,
    utilizando a lista de stopwords do NLTK para filtragem automática.
    """
    tokens = nltk.word_tokenize(text.lower())

    english_stopwords = set(stopwords.words('english'))

    tagged_words = nltk.pos_tag(tokens, tagset='universal')
    key_terms = []

    single_words = [word for word, tag in tagged_words if tag in ('NOUN', 'VERB', 'ADJ') and word not in english_stopwords]
    key_terms.extend(list(set(single_words)))

    bigrams = nltk.bigrams(tokens)
    for bigram in bigrams:
        if bigram[0] not in english_stopwords and bigram[1] not in english_stopwords:
            if nltk.pos_tag([bigram[0]], tagset='universal')[0][1] in ('ADJ', 'NOUN') and \
               nltk.pos_tag([bigram[1]], tagset='universal')[0][1] in ('NOUN'):
                key_terms.append(' '.join(bigram))

    return list(set(key_terms))

def extract_entities_from_argument_and_topic(question_prompt):
    try:
        parts = question_prompt.split('SUPPORT or ATTACK the belief')
        argument_text = parts[0].replace('The argument', '').strip().strip('"')
        topic_text = parts[1].strip().strip('"?')

        argument_entities = extract_key_terms(argument_text)
        topic_entities = extract_key_terms(topic_text)

        return argument_entities, topic_entities
    except Exception as e:
        print(f"Erro ao parsear o prompt: {e}")
        return [], []

async def fetch_paths(session, semaphore, rate_limiter, arg_entity, topic_entity):
    """
    Função assíncrona para buscar caminhos entre duas entidades, usando semáforo e limitador de taxa.
    """
    if arg_entity == topic_entity:
        return []

    async with semaphore:
        await rate_limiter.wait()
        url = f"http://api.conceptnet.io/query?start=/c/en/{arg_entity.replace(' ', '_')}&end=/c/en/{topic_entity.replace(' ', '_')}"

        try:
            async with session.get(url) as response:
                response.raise_for_status()
                data = await response.json()

                paths = []
                for edge in data.get('edges', []):
                    start_label = edge['start']['label']
                    end_label = edge['end']['label']
                    rel_label = edge['rel']['label']

                    if rel_label not in excluded_relations and start_label != end_label:
                        paths.append(f"'{start_label}' -> ({rel_label}) -> '{end_label}'")
                return paths

        except (aiohttp.ClientError, asyncio.TimeoutError, JSONDecodeError) as e:
            print(f"Erro na requisição para a API do ConceptNet entre '{arg_entity}' e '{topic_entity}': {e}")
            return []

async def find_paths_between_entities_online_async(argument_entities, topic_entities):
    """
    Busca por caminhos entre entidades usando a API online do ConceptNet
    de forma assíncrona para maior velocidade.
    """
    paths_list = []

    all_entities = list(set(argument_entities + topic_entities))

    semaphore = asyncio.Semaphore(5)
    rate_limiter = RateLimiter(max_requests=120, period=60)

    tasks = [asyncio.ensure_future(fetch_paths(session, semaphore, rate_limiter, arg_entity, topic_entity))
             for arg_entity in all_entities for topic_entity in all_entities]

    results = await asyncio.gather(*tasks)

    for result_paths in results:
        paths_list.extend(result_paths)

    if paths_list:
        return ' '.join(sorted(list(set(paths_list))))
    else:
        return 'Nenhum caminho direto encontrado entre os conceitos do argumento e do tópico.'

async def create_prompt_with_paths_online(question_prompt):
    """
    Função assíncrona para criar o prompt completo.
    """
    argument_entities, topic_entities = extract_entities_from_argument_and_topic(question_prompt)

    argument_entities = list(set(argument_entities) - set(topic_entities))

    conceptnet_context = await find_paths_between_entities_online_async(argument_entities, topic_entities)

    system_instruction = ("Your task is to analyze an argument and a belief, using the provided 'Knowledge Graph Context' "
                          "which shows direct paths between concepts from the argument and the belief. "
                          "Based on this, decide if the argument attacks or supports the belief. "
                          "Your answer must be ONLY the word 'ATTACK' or 'SUPPORT'.")

    full_prompt = (f"{system_instruction}\\n\\n"
                   f"Knowledge Graph Context: {conceptnet_context}\\n\\n"
                   f"Question: {question_prompt}")

    return full_prompt

# ==============================================================================
# CÉLULA 7: TESTE DE CASO ÚNICO ONLINE
# ==============================================================================
print("--- Executando teste de caso único para medir o tempo de criação do prompt ---")

single_prompt_text = (
    'The argument "Exposure to violent video games causes at least a temporary increase in aggression and this exposure correlates with aggression in the real world" SUPPORT or ATTACK the belief "This house believes that the sale of violent video games to minors should be banned"?'
)

async def main():
    start_time = time.time()
    prompt_gerado = await create_prompt_with_paths_online(single_prompt_text)
    end_time = time.time()

    print(f"\n⏱️ Tempo de execução para gerar o prompt: {end_time - start_time:.4f} segundos")
    print("\n--- Prompt Gerado ---")
    print(prompt_gerado)

# Chamada principal para executar a função assíncrona
# Em um notebook Colab, basta usar 'await' na chamada da função async
await main()

In [ ]:
from conceptnet_lite import Label

In [ ]:
from peewee import OperationalError

In [ ]:
# ==============================================================================
# CÉLULA 7: TESTE DE CASO ÚNICO
# ==============================================================================
print("--- Executando teste de caso único para medir o tempo de criação do prompt ---")

import time

print("--- Executando teste de caso único para medir o tempo de criação do prompt ---")

# Corrigido o uso de aspas
'''
single_prompt_text = (
    'The argument "If, for any reason, the single child is unable to care for their older adult relatives, '
    'the oldest generations would face a lack of resources and necessities" SUPPORT or ATTACK the belief '
    '"This house supports the one-child policy of the republic of China"?'
)
'''

single_prompt_text = (
    'The argument "Exposure to violent video games causes at least a temporary increase in aggression and this exposure correlates with aggression in the real world" SUPPORT or ATTACK the belief "This house believes that the sale of violent video games to minors should be banned"?'
)

start_time = time.time()
prompt_gerado = create_prompt_with_paths_local(single_prompt_text)
end_time = time.time()

print(f"\n⏱️ Tempo de execução para gerar o prompt: {end_time - start_time:.4f} segundos")
print("\n--- Prompt Gerado ---")
print(prompt_gerado)

In [ ]:
# ==============================================================================
# CÉLULA 7: TESTE DE CASO ÚNICO ONLINE
# ==============================================================================
print("--- Executando teste de caso único para medir o tempo de criação do prompt ---")

import time

print("--- Executando teste de caso único para medir o tempo de criação do prompt ---")

# Corrigido o uso de aspas
'''
single_prompt_text = (
    'The argument "If, for any reason, the single child is unable to care for their older adult relatives, '
    'the oldest generations would face a lack of resources and necessities" SUPPORT or ATTACK the belief '
    '"This house supports the one-child policy of the republic of China"?'
)
'''

single_prompt_text = (
    'The argument "Exposure to violent video games causes at least a temporary increase in aggression and this exposure correlates with aggression in the real world" SUPPORT or ATTACK the belief "This house believes that the sale of violent video games to minors should be banned"?'
)

start_time = time.time()
prompt_gerado = create_prompt_with_paths_online(single_prompt_text)
end_time = time.time()

print(f"\n⏱️ Tempo de execução para gerar o prompt: {end_time - start_time:.4f} segundos")
print("\n--- Prompt Gerado ---")
print(prompt_gerado)

In [ ]:
# ==============================================================================
# CÉLULA 8: PROCESSAMENTO EM LOTE COM CHECKPOINTING
# ==============================================================================
prompts_filename = '/content/drive/My Drive/.../prompts_IBM.csv'
processed_prompts = set()

# Verifica se o arquivo já existe e carrega os prompts já processados
if os.path.exists(prompts_filename):
    print(f"Arquivo '{prompts_filename}' encontrado. Carregando prompts já processados...")
    df_existente = pd.read_csv(prompts_filename)
    processed_prompts = set(df_existente['prompt_question'])
    print(f"{len(processed_prompts)} prompts já haviam sido processados.")
else:
    print(f"Arquivo '{prompts_filename}' não encontrado. Um novo será criado.")

# Filtra o dataframe para processar apenas os prompts que ainda não estão no arquivo
prompts_a_processar = test_df[~test_df['question'].isin(processed_prompts)]

if prompts_a_processar.empty:
    print("Todos os prompts já foram processados. Nenhuma ação necessária.")
else:
    print(f"Iniciando o processamento de {len(prompts_a_processar)} novos prompts...")

    # Itera sobre os prompts restantes
    for index, row in prompts_a_processar.iterrows():
        start_time_prompt = time.time()

        question = row['question']
        right_answer = row['claims.stance']

        # Cria o prompt enriquecido
        full_prompt = create_prompt_with_paths_local(question)

        # Prepara os dados para salvar
        novo_prompt_df = pd.DataFrame([{
            'prompt_question': question, # A pergunta original
            'full_prompt_text': full_prompt, # O prompt completo com contexto
            'right_answer': right_answer
        }])

        # Anexa ao arquivo CSV
        # O cabeçalho só é escrito se o arquivo não existir
        header = not os.path.exists(prompts_filename)
        novo_prompt_df.to_csv(prompts_filename, mode='a', header=header, index=False)

        end_time_prompt = time.time()
        print(f"Prompt {index} processado e salvo em {end_time_prompt - start_time_prompt:.2f}s.")

print("Processamento em lote concluído.")

In [ ]:
import sys
import subprocess
import os

# --- Bloco de instalação de bibliotecas ---
print("Verificando e instalando bibliotecas necessárias...")
required_packages = ["aiohttp", "pandas", "nltk", "nest_asyncio"]
for package in required_packages:
    try:
        __import__(package)
        print(f"Biblioteca '{package}' já instalada.")
    except ImportError:
        print(f"Biblioteca '{package}' não encontrada. Instalando...")
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", package])
            print(f"Biblioteca '{package}' instalada com sucesso.")
        except subprocess.CalledProcessError as e:
            print(f"Erro ao instalar a biblioteca '{package}': {e}")
            sys.exit(1)

# Recarrega as bibliotecas após a instalação, se necessário
import aiohttp
import pandas as pd
import nltk
import nest_asyncio

# Aplica o patch para permitir loops de eventos aninhados, resolvendo o RuntimeError
nest_asyncio.apply()

# Baixa a lista de stopwords do NLTK, se ainda não estiver instalada
try:
    nltk.data.find('corpora/stopwords')
    print("NLTK stopwords já baixadas.")
except nltk.downloader.DownloadError:
    print("Baixando NLTK stopwords...")
    nltk.download('stopwords')

import asyncio
from requests.exceptions import RequestException, JSONDecodeError
from nltk.corpus import stopwords
import time
import random

# Placeholder para o seu DataFrame, caso ele não tenha sido carregado
try:
    # Tenta usar o DataFrame do ambiente do usuário
    test_df
except NameError:
    print("\nAviso: O DataFrame 'test_df' não foi encontrado. Criando um DataFrame vazio para demonstração.")
    # Exemplo de como você poderia carregar seu DataFrame, se necessário
    # test_df = pd.read_csv('caminho_para_o_seu_arquivo_ibm.csv')
    test_df = pd.DataFrame(columns=['question', 'claims.stance'])
else:
    print("DataFrame 'test_df' encontrado no ambiente.")


# ==============================================================================
# CÉLULA 6: FUNÇÕES ADAPTADAS PARA CONCEPTNET ONLINE (ASSÍNCRONAS COM LIMITADOR)
# ==============================================================================
# Relações a serem excluídas
excluded_relations = {'AtLocation', 'EtymologicallyRelated'}

def extract_key_terms(text):
    """
    Extrai termos-chave, incluindo frases de substantivos (chunks), para capturar conceitos mais complexos,
    utilizando a lista de stopwords do NLTK para filtragem automática.
    """
    tokens = nltk.word_tokenize(text.lower())

    english_stopwords = set(stopwords.words('english'))

    tagged_words = nltk.pos_tag(tokens, tagset='universal')
    key_terms = []

    single_words = [word for word, tag in tagged_words if tag in ('NOUN', 'VERB', 'ADJ') and word not in english_stopwords]
    key_terms.extend(list(set(single_words)))

    bigrams = nltk.bigrams(tokens)
    for bigram in bigrams:
        if bigram[0] not in english_stopwords and bigram[1] not in english_stopwords:
            if nltk.pos_tag([bigram[0]], tagset='universal')[0][1] in ('ADJ', 'NOUN') and \
               nltk.pos_tag([bigram[1]], tagset='universal')[0][1] in ('NOUN'):
                key_terms.append(' '.join(bigram))

    return list(set(key_terms))

def extract_entities_from_argument_and_topic(question_prompt):
    try:
        parts = question_prompt.split('SUPPORT or ATTACK the belief')
        argument_text = parts[0].replace('The argument', '').strip().strip('"')
        topic_text = parts[1].strip().strip('"?')

        argument_entities = extract_key_terms(argument_text)
        topic_entities = extract_key_terms(topic_text)

        return argument_entities, topic_entities
    except Exception as e:
        print(f"Erro ao parsear o prompt: {e}")
        return [], []

async def fetch_paths(session, semaphore, rate_limiter, arg_entity, topic_entity):
    """
    Função assíncrona para buscar caminhos entre duas entidades, usando semáforo e limitador de taxa.
    """
    if arg_entity == topic_entity:
        return []

    async with semaphore:
        await rate_limiter.wait()
        url = f"http://api.conceptnet.io/query?start=/c/en/{arg_entity.replace(' ', '_')}&end=/c/en/{topic_entity.replace(' ', '_')}"

        try:
            async with session.get(url) as response:
                response.raise_for_status()
                data = await response.json()

                paths = []
                for edge in data.get('edges', []):
                    start_label = edge['start']['label']
                    end_label = edge['end']['label']
                    rel_label = edge['rel']['label']

                    if rel_label not in excluded_relations and start_label != end_label:
                        paths.append(f"'{start_label}' -> ({rel_label}) -> '{end_label}'")
                return paths

        except (aiohttp.ClientError, asyncio.TimeoutError, JSONDecodeError) as e:
            print(f"Erro na requisição para a API do ConceptNet entre '{arg_entity}' e '{topic_entity}': {e}")
            return []

async def find_paths_between_entities_online_async(argument_entities, topic_entities):
    """
    Busca por caminhos entre entidades usando a API online do ConceptNet
    de forma assíncrona para maior velocidade.
    """
    paths_list = []

    all_entities = list(set(argument_entities + topic_entities))

    semaphore = asyncio.Semaphore(5)
    rate_limiter = RateLimiter(max_requests=120, period=60)

    tasks = [asyncio.ensure_future(fetch_paths(session, semaphore, rate_limiter, arg_entity, topic_entity))
             for arg_entity in all_entities for topic_entity in all_entities]

    results = await asyncio.gather(*tasks)

    for result_paths in results:
        paths_list.extend(result_paths)

    if paths_list:
        return ' '.join(sorted(list(set(paths_list))))
    else:
        return 'Nenhum caminho direto encontrado entre os conceitos do argumento e do tópico.'

async def create_prompt_with_paths_online(question_prompt):
    """
    Função assíncrona para criar o prompt completo.
    """
    argument_entities, topic_entities = extract_entities_from_argument_and_topic(question_prompt)

    argument_entities = list(set(argument_entities) - set(topic_entities))

    conceptnet_context = await find_paths_between_entities_online_async(argument_entities, topic_entities)

    system_instruction = ("Your task is to analyze an argument and a belief, using the provided 'Knowledge Graph Context' "
                          "which shows direct paths between concepts from the argument and the belief. "
                          "Based on this, decide if the argument attacks or supports the belief. "
                          "Your answer must be ONLY the word 'ATTACK' or 'SUPPORT'.")

    full_prompt = (f"{system_instruction}\\n\\n"
                   f"Knowledge Graph Context: {conceptnet_context}\\n\\n"
                   f"Question: {question_prompt}")

    return full_prompt

# ==============================================================================
# CÉLULA 8: PROCESSAMENTO EM LOTE COM CHECKPOINTING (ASSÍNCRONO)
# ==============================================================================
prompts_filename = '/content/drive/My Drive/.../prompts_IBM.csv'

async def batch_process_main(prompts_to_process, filename):
    total_requests_made = 0
    prompts_processed_count = 0
    start_time_batch = time.perf_counter()

    if prompts_to_process.empty:
        print("Todos os prompts já foram processados. Nenhuma ação necessária.")
        return

    print(f"\nIniciando o processamento de {len(prompts_to_process)} novos prompts de forma assíncrona...")

    async with aiohttp.ClientSession() as session:
        header = not os.path.exists(filename)

        for index, row in prompts_to_process.iterrows():
            question = row['question']
            right_answer = row['claims.stance']

            try:
                full_prompt, requests_count = await create_prompt_with_paths_online(question, session)

                novo_prompt_df = pd.DataFrame([{
                    'prompt_question': question,
                    'full_prompt_text': full_prompt,
                    'right_answer': right_answer
                }])

                novo_prompt_df.to_csv(filename, mode='a', header=header if prompts_processed_count == 0 else False, index=False)

                prompts_processed_count += 1
                total_requests_made += requests_count

                current_time = time.perf_counter()
                elapsed_time = current_time - start_time_batch

                print(f"--- Progresso do Prompt {prompts_processed_count} ---")
                print(f"Tempo decorrido: {elapsed_time:.2f} segundos")
                print(f"Requisições à API do ConceptNet feitas para este prompt: {requests_count}")
                print(f"Total de prompts processados: {prompts_processed_count}/{len(prompts_to_process)}")
                print(f"Total de requisições acumuladas: {total_requests_made}")
                print("-" * 30)

            except Exception as e:
                print(f"Erro ao processar o prompt: {question}. Erro: {e}")
                continue

    end_time_batch = time.perf_counter()
    print(f"\nProcessamento em lote concluído em {end_time_batch - start_time_batch:.2f}s.")
    print(f"Total de prompts processados: {prompts_processed_count}")
    print(f"Total de requisições feitas: {total_requests_made}")


async def main_execution():
    if os.path.exists(prompts_filename):
        print(f"Arquivo '{prompts_filename}' encontrado. Carregando prompts já processados...")
        df_existente = pd.read_csv(prompts_filename)
        processed_prompts = set(df_existente['prompt_question'])
        print(f"{len(processed_prompts)} prompts já haviam sido processados.")
    else:
        print(f"Arquivo '{prompts_filename}' não encontrado. Um novo será criado.")
        processed_prompts = set()

    if 'test_df' not in locals() and 'test_df' not in globals():
        print("Erro: DataFrame 'test_df' não encontrado. Por favor, carregue o seu DataFrame de teste.")
        prompts_a_processar = pd.DataFrame()
    else:
        prompts_a_processar = test_df[~test_df['question'].isin(processed_prompts)]

    await batch_process_main(prompts_a_processar, prompts_filename)

def run_main():
    try:
        asyncio.run(main_execution())
    except RuntimeError as e:
        if "Event loop is already running" in str(e):
            print("Event loop já está em execução. Usando nest_asyncio...")
            loop = asyncio.get_event_loop()
            if loop.is_running():
                loop.run_until_complete(main_execution())
            else:
                loop.run_until_complete(main_execution())
        else:
            raise e

# --- Bloco de execução principal ---
if __name__ == "__main__":
    run_main()

In [ ]:
# ==============================================================================
# CÉLULA 6: FUNÇÕES ADAPTADAS PARA O GOOGLE KNOWLEDGE GRAPH (ASSÍNCRONAS)
# ==============================================================================
import nltk
import requests
import asyncio
import aiohttp
import json
from requests.exceptions import RequestException, JSONDecodeError
from nltk.corpus import stopwords
import time

# Baixa a lista de stopwords do NLTK, se ainda não estiver instalada
try:
    nltk.data.find('corpora/stopwords')
except nltk.downloader.DownloadError:
    nltk.download('stopwords')

# --- Insira sua chave de API do Google aqui ---
# Exemplo: api_key = ''
api_key = ''

def extract_key_terms(text):
    """
    Extrai termos-chave, incluindo frases de substantivos (chunks), para capturar conceitos mais complexos,
    utilizando a lista de stopwords do NLTK para filtragem automática.
    """
    tokens = nltk.word_tokenize(text.lower())

    english_stopwords = set(stopwords.words('english'))

    tagged_words = nltk.pos_tag(tokens, tagset='universal')
    key_terms = []

    single_words = [word for word, tag in tagged_words if tag in ('NOUN', 'VERB', 'ADJ') and word not in english_stopwords]
    key_terms.extend(list(set(single_words)))

    bigrams = nltk.bigrams(tokens)
    for bigram in bigrams:
        if bigram[0] not in english_stopwords and bigram[1] not in english_stopwords:
            if nltk.pos_tag([bigram[0]], tagset='universal')[0][1] in ('ADJ', 'NOUN') and \
               nltk.pos_tag([bigram[1]], tagset='universal')[0][1] in ('NOUN'):
                key_terms.append(' '.join(bigram))

    return list(set(key_terms))

def extract_entities_from_argument_and_topic(question_prompt):
    try:
        parts = question_prompt.split('SUPPORT or ATTACK the belief')
        argument_text = parts[0].replace('The argument', '').strip().strip('"')
        topic_text = parts[1].strip().strip('"?')

        argument_entities = extract_key_terms(argument_text)
        topic_entities = extract_key_terms(topic_text)

        return argument_entities, topic_entities
    except Exception as e:
        print(f"Erro ao parsear o prompt: {e}")
        return [], []

async def fetch_entity_info(session, entity):
    """
    Função assíncrona para buscar informações de uma entidade na API do Google Knowledge Graph.
    """
    if not api_key:
        print("Erro: A chave da API não foi definida. Por favor, insira sua chave.")
        return f"Erro na API para o termo '{entity}'."

    # A API do Google é focada em busca, então usamos um termo por vez
    url = f"https://kgsearch.googleapis.com/v1/entities:search?query={entity.replace(' ', '%20')}&key={api_key}"

    try:
        async with session.get(url) as response:
            response.raise_for_status()
            data = await response.json()

            # Pega o primeiro resultado mais relevante
            if data and 'itemListElement' in data and data['itemListElement']:
                result = data['itemListElement'][0]['result']
                name = result.get('name', 'N/A')
                description = result.get('description', 'N/A')

                return f"Conceito '{name}' (tipo: {', '.join(result.get('@type', []))}) - Descrição: {description}."
            else:
                return f"Nenhuma informação encontrada para o termo '{entity}'."

    except (aiohttp.ClientError, asyncio.TimeoutError, JSONDecodeError) as e:
        print(f"Erro na requisição para a API do Google Knowledge Graph para o termo '{entity}': {e}")
        return f"Erro na API para o termo '{entity}'."

async def find_paths_between_entities_online_async(argument_entities, topic_entities):
    """
    Busca por informações das entidades usando a API do Google de forma assíncrona.
    """
    all_entities = list(set(argument_entities + topic_entities))

    async with aiohttp.ClientSession() as session:
        # Cria uma lista de tarefas para buscar cada entidade
        tasks = [asyncio.ensure_future(fetch_entity_info(session, entity))
                 for entity in all_entities]

        results = await asyncio.gather(*tasks)

    # Combina os resultados em uma única string de contexto
    if any("Nenhuma informação encontrada" not in r and "Erro na API" not in r for r in results):
        return ' '.join(list(set(results)))
    else:
        return 'Nenhum contexto relevante encontrado para os conceitos do argumento e do tópico.'

async def create_prompt_with_paths_online(question_prompt):
    """
    Função assíncrona para criar o prompt completo.
    """
    argument_entities, topic_entities = extract_entities_from_argument_and_topic(question_prompt)

    # Busca informações para todas as entidades
    conceptnet_context = await find_paths_between_entities_online_async(argument_entities, topic_entities)

    system_instruction = ("Your task is to analyze an argument and a belief, using the provided 'Knowledge Graph Context' "
                          "which shows direct paths between concepts from the argument and the belief. "
                          "Based on this, decide if the argument attacks or supports the belief. "
                          "Your answer must be ONLY the word 'ATTACK' or 'SUPPORT'.")

    full_prompt = (f"{system_instruction}\\n\\n"
                   f"Knowledge Graph Context: {conceptnet_context}\\n\\n"
                   f"Question: {question_prompt}")

    return full_prompt

# ==============================================================================
# CÉLULA 7: TESTE DE CASO ÚNICO ONLINE
# ==============================================================================
print("--- Executando teste de caso único para medir o tempo de criação do prompt ---")

single_prompt_text = (
    'The argument "Exposure to violent video games causes at least a temporary increase in aggression and this exposure correlates with aggression in the real world" SUPPORT or ATTACK the belief "This house believes that the sale of violent video games to minors should be banned"?'
)

async def main():
    start_time = time.time()
    prompt_gerado = await create_prompt_with_paths_online(single_prompt_text)
    end_time = time.time()

    print(f"\n⏱️ Tempo de execução para gerar o prompt: {end_time - start_time:.4f} segundos")
    print("\n--- Prompt Gerado ---")
    print(prompt_gerado)

# Chamada principal para executar a função assíncrona
# Em um notebook Colab, basta usar 'await' na chamada da função async
await main()

In [ ]:
from conceptnet_lite import Label

In [ ]:
from peewee import OperationalError

In [ ]:
# ==============================================================================
# CÉLULA 7: TESTE DE CASO ÚNICO
# ==============================================================================
print("--- Executando teste de caso único para medir o tempo de criação do prompt ---")

import time

print("--- Executando teste de caso único para medir o tempo de criação do prompt ---")

# Corrigido o uso de aspas
'''
single_prompt_text = (
    'The argument "If, for any reason, the single child is unable to care for their older adult relatives, '
    'the oldest generations would face a lack of resources and necessities" SUPPORT or ATTACK the belief '
    '"This house supports the one-child policy of the republic of China"?'
)
'''

single_prompt_text = (
    'The argument "Exposure to violent video games causes at least a temporary increase in aggression and this exposure correlates with aggression in the real world" SUPPORT or ATTACK the belief "This house believes that the sale of violent video games to minors should be banned"?'
)

start_time = time.time()
prompt_gerado = create_prompt_with_paths_local(single_prompt_text)
end_time = time.time()

print(f"\n⏱️ Tempo de execução para gerar o prompt: {end_time - start_time:.4f} segundos")
print("\n--- Prompt Gerado ---")
print(prompt_gerado)

In [ ]:
# ==============================================================================
# CÉLULA 7: TESTE DE CASO ÚNICO ONLINE
# ==============================================================================
print("--- Executando teste de caso único para medir o tempo de criação do prompt ---")

import time

print("--- Executando teste de caso único para medir o tempo de criação do prompt ---")

# Corrigido o uso de aspas
'''
single_prompt_text = (
    'The argument "If, for any reason, the single child is unable to care for their older adult relatives, '
    'the oldest generations would face a lack of resources and necessities" SUPPORT or ATTACK the belief '
    '"This house supports the one-child policy of the republic of China"?'
)
'''

single_prompt_text = (
    'The argument "Exposure to violent video games causes at least a temporary increase in aggression and this exposure correlates with aggression in the real world" SUPPORT or ATTACK the belief "This house believes that the sale of violent video games to minors should be banned"?'
)

start_time = time.time()
prompt_gerado = create_prompt_with_paths_online(single_prompt_text)
end_time = time.time()

print(f"\n⏱️ Tempo de execução para gerar o prompt: {end_time - start_time:.4f} segundos")
print("\n--- Prompt Gerado ---")
print(prompt_gerado)

In [ ]:
# ==============================================================================
# CÉLULA 8: PROCESSAMENTO EM LOTE COM CHECKPOINTING
# ==============================================================================
prompts_filename = '/content/drive/My Drive/.../prompts_IBM.csv'
processed_prompts = set()

# Verifica se o arquivo já existe e carrega os prompts já processados
if os.path.exists(prompts_filename):
    print(f"Arquivo '{prompts_filename}' encontrado. Carregando prompts já processados...")
    df_existente = pd.read_csv(prompts_filename)
    processed_prompts = set(df_existente['prompt_question'])
    print(f"{len(processed_prompts)} prompts já haviam sido processados.")
else:
    print(f"Arquivo '{prompts_filename}' não encontrado. Um novo será criado.")

# Filtra o dataframe para processar apenas os prompts que ainda não estão no arquivo
prompts_a_processar = test_df[~test_df['question'].isin(processed_prompts)]

if prompts_a_processar.empty:
    print("Todos os prompts já foram processados. Nenhuma ação necessária.")
else:
    print(f"Iniciando o processamento de {len(prompts_a_processar)} novos prompts...")

    # Itera sobre os prompts restantes
    for index, row in prompts_a_processar.iterrows():
        start_time_prompt = time.time()

        question = row['question']
        right_answer = row['claims.stance']

        # Cria o prompt enriquecido
        full_prompt = create_prompt_with_paths_local(question)

        # Prepara os dados para salvar
        novo_prompt_df = pd.DataFrame([{
            'prompt_question': question, # A pergunta original
            'full_prompt_text': full_prompt, # O prompt completo com contexto
            'right_answer': right_answer
        }])

        # Anexa ao arquivo CSV
        # O cabeçalho só é escrito se o arquivo não existir
        header = not os.path.exists(prompts_filename)
        novo_prompt_df.to_csv(prompts_filename, mode='a', header=header, index=False)

        end_time_prompt = time.time()
        print(f"Prompt {index} processado e salvo em {end_time_prompt - start_time_prompt:.2f}s.")

print("Processamento em lote concluído.")

In [ ]:
import sys
import subprocess
import os

# --- Bloco de instalação de bibliotecas ---
print("Verificando e instalando bibliotecas necessárias...")
required_packages = ["aiohttp", "pandas", "nltk", "nest_asyncio"]
for package in required_packages:
    try:
        __import__(package)
        print(f"Biblioteca '{package}' já instalada.")
    except ImportError:
        print(f"Biblioteca '{package}' não encontrada. Instalando...")
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", package])
            print(f"Biblioteca '{package}' instalada com sucesso.")
        except subprocess.CalledProcessError as e:
            print(f"Erro ao instalar a biblioteca '{package}': {e}")
            sys.exit(1)

# Recarrega as bibliotecas após a instalação, se necessário
import aiohttp
import pandas as pd
import nltk
import nest_asyncio

# Aplica o patch para permitir loops de eventos aninhados, resolvendo o RuntimeError
nest_asyncio.apply()

# Baixa a lista de stopwords do NLTK, se ainda não estiver instalada
try:
    nltk.data.find('corpora/stopwords')
    print("NLTK stopwords já baixadas.")
except nltk.downloader.DownloadError:
    print("Baixando NLTK stopwords...")
    nltk.download('stopwords')

import asyncio
from requests.exceptions import RequestException, JSONDecodeError
from nltk.corpus import stopwords
import time
import random

# Placeholder para o seu DataFrame, caso ele não tenha sido carregado
try:
    # Tenta usar o DataFrame do ambiente do usuário
    test_df
except NameError:
    print("\nAviso: O DataFrame 'test_df' não foi encontrado. Criando um DataFrame vazio para demonstração.")
    # Exemplo de como você poderia carregar seu DataFrame, se necessário
    # test_df = pd.read_csv('caminho_para_o_seu_arquivo_ibm.csv')
    test_df = pd.DataFrame(columns=['question', 'claims.stance'])
else:
    print("DataFrame 'test_df' encontrado no ambiente.")


# ==============================================================================
# CÉLULA 6: FUNÇÕES ADAPTADAS PARA CONCEPTNET ONLINE (ASSÍNCRONAS COM LIMITADOR)
# ==============================================================================
# Relações a serem excluídas
excluded_relations = {'AtLocation', 'EtymologicallyRelated'}

def extract_key_terms(text):
    """
    Extrai termos-chave, incluindo frases de substantivos (chunks), para capturar conceitos mais complexos,
    utilizando a lista de stopwords do NLTK para filtragem automática.
    """
    tokens = nltk.word_tokenize(text.lower())

    english_stopwords = set(stopwords.words('english'))

    tagged_words = nltk.pos_tag(tokens, tagset='universal')
    key_terms = []

    single_words = [word for word, tag in tagged_words if tag in ('NOUN', 'VERB', 'ADJ') and word not in english_stopwords]
    key_terms.extend(list(set(single_words)))

    bigrams = nltk.bigrams(tokens)
    for bigram in bigrams:
        if bigram[0] not in english_stopwords and bigram[1] not in english_stopwords:
            if nltk.pos_tag([bigram[0]], tagset='universal')[0][1] in ('ADJ', 'NOUN') and \
               nltk.pos_tag([bigram[1]], tagset='universal')[0][1] in ('NOUN'):
                key_terms.append(' '.join(bigram))

    return list(set(key_terms))

def extract_entities_from_argument_and_topic(question_prompt):
    try:
        parts = question_prompt.split('SUPPORT or ATTACK the belief')
        argument_text = parts[0].replace('The argument', '').strip().strip('"')
        topic_text = parts[1].strip().strip('"?')

        argument_entities = extract_key_terms(argument_text)
        topic_entities = extract_key_terms(topic_text)

        return argument_entities, topic_entities
    except Exception as e:
        print(f"Erro ao parsear o prompt: {e}")
        return [], []

async def fetch_paths(session, semaphore, rate_limiter, arg_entity, topic_entity):
    """
    Função assíncrona para buscar caminhos entre duas entidades, usando semáforo e limitador de taxa.
    """
    if arg_entity == topic_entity:
        return []

    async with semaphore:
        await rate_limiter.wait()
        url = f"http://api.conceptnet.io/query?start=/c/en/{arg_entity.replace(' ', '_')}&end=/c/en/{topic_entity.replace(' ', '_')}"

        try:
            async with session.get(url) as response:
                response.raise_for_status()
                data = await response.json()

                paths = []
                for edge in data.get('edges', []):
                    start_label = edge['start']['label']
                    end_label = edge['end']['label']
                    rel_label = edge['rel']['label']

                    if rel_label not in excluded_relations and start_label != end_label:
                        paths.append(f"'{start_label}' -> ({rel_label}) -> '{end_label}'")
                return paths

        except (aiohttp.ClientError, asyncio.TimeoutError, JSONDecodeError) as e:
            print(f"Erro na requisição para a API do ConceptNet entre '{arg_entity}' e '{topic_entity}': {e}")
            return []

async def find_paths_between_entities_online_async(argument_entities, topic_entities):
    """
    Busca por caminhos entre entidades usando a API online do ConceptNet
    de forma assíncrona para maior velocidade.
    """
    paths_list = []

    all_entities = list(set(argument_entities + topic_entities))

    semaphore = asyncio.Semaphore(5)
    rate_limiter = RateLimiter(max_requests=120, period=60)

    tasks = [asyncio.ensure_future(fetch_paths(session, semaphore, rate_limiter, arg_entity, topic_entity))
             for arg_entity in all_entities for topic_entity in all_entities]

    results = await asyncio.gather(*tasks)

    for result_paths in results:
        paths_list.extend(result_paths)

    if paths_list:
        return ' '.join(sorted(list(set(paths_list))))
    else:
        return 'Nenhum caminho direto encontrado entre os conceitos do argumento e do tópico.'

async def create_prompt_with_paths_online(question_prompt):
    """
    Função assíncrona para criar o prompt completo.
    """
    argument_entities, topic_entities = extract_entities_from_argument_and_topic(question_prompt)

    argument_entities = list(set(argument_entities) - set(topic_entities))

    conceptnet_context = await find_paths_between_entities_online_async(argument_entities, topic_entities)

    system_instruction = ("Your task is to analyze an argument and a belief, using the provided 'Knowledge Graph Context' "
                          "which shows direct paths between concepts from the argument and the belief. "
                          "Based on this, decide if the argument attacks or supports the belief. "
                          "Your answer must be ONLY the word 'ATTACK' or 'SUPPORT'.")

    full_prompt = (f"{system_instruction}\\n\\n"
                   f"Knowledge Graph Context: {conceptnet_context}\\n\\n"
                   f"Question: {question_prompt}")

    return full_prompt

# ==============================================================================
# CÉLULA 8: PROCESSAMENTO EM LOTE COM CHECKPOINTING (ASSÍNCRONO)
# ==============================================================================
prompts_filename = '/content/drive/My Drive/.../prompts_IBM.csv'

async def batch_process_main(prompts_to_process, filename):
    total_requests_made = 0
    prompts_processed_count = 0
    start_time_batch = time.perf_counter()

    if prompts_to_process.empty:
        print("Todos os prompts já foram processados. Nenhuma ação necessária.")
        return

    print(f"\nIniciando o processamento de {len(prompts_to_process)} novos prompts de forma assíncrona...")

    async with aiohttp.ClientSession() as session:
        header = not os.path.exists(filename)

        for index, row in prompts_to_process.iterrows():
            question = row['question']
            right_answer = row['claims.stance']

            try:
                full_prompt, requests_count = await create_prompt_with_paths_online(question, session)

                novo_prompt_df = pd.DataFrame([{
                    'prompt_question': question,
                    'full_prompt_text': full_prompt,
                    'right_answer': right_answer
                }])

                novo_prompt_df.to_csv(filename, mode='a', header=header if prompts_processed_count == 0 else False, index=False)

                prompts_processed_count += 1
                total_requests_made += requests_count

                current_time = time.perf_counter()
                elapsed_time = current_time - start_time_batch

                print(f"--- Progresso do Prompt {prompts_processed_count} ---")
                print(f"Tempo decorrido: {elapsed_time:.2f} segundos")
                print(f"Requisições à API do ConceptNet feitas para este prompt: {requests_count}")
                print(f"Total de prompts processados: {prompts_processed_count}/{len(prompts_to_process)}")
                print(f"Total de requisições acumuladas: {total_requests_made}")
                print("-" * 30)

            except Exception as e:
                print(f"Erro ao processar o prompt: {question}. Erro: {e}")
                continue

    end_time_batch = time.perf_counter()
    print(f"\nProcessamento em lote concluído em {end_time_batch - start_time_batch:.2f}s.")
    print(f"Total de prompts processados: {prompts_processed_count}")
    print(f"Total de requisições feitas: {total_requests_made}")


async def main_execution():
    if os.path.exists(prompts_filename):
        print(f"Arquivo '{prompts_filename}' encontrado. Carregando prompts já processados...")
        df_existente = pd.read_csv(prompts_filename)
        processed_prompts = set(df_existente['prompt_question'])
        print(f"{len(processed_prompts)} prompts já haviam sido processados.")
    else:
        print(f"Arquivo '{prompts_filename}' não encontrado. Um novo será criado.")
        processed_prompts = set()

    if 'test_df' not in locals() and 'test_df' not in globals():
        print("Erro: DataFrame 'test_df' não encontrado. Por favor, carregue o seu DataFrame de teste.")
        prompts_a_processar = pd.DataFrame()
    else:
        prompts_a_processar = test_df[~test_df['question'].isin(processed_prompts)]

    await batch_process_main(prompts_a_processar, prompts_filename)

def run_main():
    try:
        asyncio.run(main_execution())
    except RuntimeError as e:
        if "Event loop is already running" in str(e):
            print("Event loop já está em execução. Usando nest_asyncio...")
            loop = asyncio.get_event_loop()
            if loop.is_running():
                loop.run_until_complete(main_execution())
            else:
                loop.run_until_complete(main_execution())
        else:
            raise e

# --- Bloco de execução principal ---
if __name__ == "__main__":
    run_main()

In [ ]:
# ==============================================================================
# CÉLULA 6: FUNÇÕES ADAPTADAS PARA O GOOGLE KNOWLEDGE GRAPH (ASSÍNCRONAS)
# ==============================================================================
import nltk
import requests
import asyncio
import aiohttp
import json
from requests.exceptions import RequestException, JSONDecodeError
from nltk.corpus import stopwords
import time

# Baixa a lista de stopwords do NLTK, se ainda não estiver instalada
try:
    nltk.data.find('corpora/stopwords')
except nltk.downloader.DownloadError:
    nltk.download('stopwords')

# --- Insira sua chave de API do Google aqui ---
# Exemplo: api_key = ''
api_key = ''

def extract_key_terms(text):
    """
    Extrai termos-chave, incluindo frases de substantivos (chunks), para capturar conceitos mais complexos,
    utilizando a lista de stopwords do NLTK para filtragem automática.
    """
    tokens = nltk.word_tokenize(text.lower())

    english_stopwords = set(stopwords.words('english'))

    tagged_words = nltk.pos_tag(tokens, tagset='universal')
    key_terms = []

    single_words = [word for word, tag in tagged_words if tag in ('NOUN', 'VERB', 'ADJ') and word not in english_stopwords]
    key_terms.extend(list(set(single_words)))

    bigrams = nltk.bigrams(tokens)
    for bigram in bigrams:
        if bigram[0] not in english_stopwords and bigram[1] not in english_stopwords:
            if nltk.pos_tag([bigram[0]], tagset='universal')[0][1] in ('ADJ', 'NOUN') and \
               nltk.pos_tag([bigram[1]], tagset='universal')[0][1] in ('NOUN'):
                key_terms.append(' '.join(bigram))

    return list(set(key_terms))

def extract_entities_from_argument_and_topic(question_prompt):
    try:
        parts = question_prompt.split('SUPPORT or ATTACK the belief')
        argument_text = parts[0].replace('The argument', '').strip().strip('"')
        topic_text = parts[1].strip().strip('"?')

        argument_entities = extract_key_terms(argument_text)
        topic_entities = extract_key_terms(topic_text)

        return argument_entities, topic_entities
    except Exception as e:
        print(f"Erro ao parsear o prompt: {e}")
        return [], []

async def fetch_entity_info(session, entity):
    """
    Função assíncrona para buscar informações de uma entidade na API do Google Knowledge Graph.
    """
    if not api_key:
        print("Erro: A chave da API não foi definida. Por favor, insira sua chave.")
        return f"Erro na API para o termo '{entity}'."

    # A API do Google é focada em busca, então usamos um termo por vez
    url = f"https://kgsearch.googleapis.com/v1/entities:search?query={entity.replace(' ', '%20')}&key={api_key}"

    try:
        async with session.get(url) as response:
            response.raise_for_status()
            data = await response.json()

            # Pega o primeiro resultado mais relevante
            if data and 'itemListElement' in data and data['itemListElement']:
                result = data['itemListElement'][0]['result']
                name = result.get('name', 'N/A')
                description = result.get('description', 'N/A')

                return f"Conceito '{name}' (tipo: {', '.join(result.get('@type', []))}) - Descrição: {description}."
            else:
                return f"Nenhuma informação encontrada para o termo '{entity}'."

    except (aiohttp.ClientError, asyncio.TimeoutError, JSONDecodeError) as e:
        print(f"Erro na requisição para a API do Google Knowledge Graph para o termo '{entity}': {e}")
        return f"Erro na API para o termo '{entity}'."

async def find_paths_between_entities_online_async(argument_entities, topic_entities):
    """
    Busca por informações das entidades usando a API do Google de forma assíncrona.
    """
    all_entities = list(set(argument_entities + topic_entities))

    async with aiohttp.ClientSession() as session:
        # Cria uma lista de tarefas para buscar cada entidade
        tasks = [asyncio.ensure_future(fetch_entity_info(session, entity))
                 for entity in all_entities]

        results = await asyncio.gather(*tasks)

    # Combina os resultados em uma única string de contexto
    if any("Nenhuma informação encontrada" not in r and "Erro na API" not in r for r in results):
        return ' '.join(list(set(results)))
    else:
        return 'Nenhum contexto relevante encontrado para os conceitos do argumento e do tópico.'

async def create_prompt_with_paths_online(question_prompt):
    """
    Função assíncrona para criar o prompt completo.
    """
    argument_entities, topic_entities = extract_entities_from_argument_and_topic(question_prompt)

    # Busca informações para todas as entidades
    conceptnet_context = await find_paths_between_entities_online_async(argument_entities, topic_entities)

    system_instruction = ("Your task is to analyze an argument and a belief, using the provided 'Knowledge Graph Context' "
                          "which shows direct paths between concepts from the argument and the belief. "
                          "Based on this, decide if the argument attacks or supports the belief. "
                          "Your answer must be ONLY the word 'ATTACK' or 'SUPPORT'.")

    full_prompt = (f"{system_instruction}\\n\\n"
                   f"Knowledge Graph Context: {conceptnet_context}\\n\\n"
                   f"Question: {question_prompt}")

    return full_prompt

# ==============================================================================
# CÉLULA 7: TESTE DE CASO ÚNICO ONLINE
# ==============================================================================
print("--- Executando teste de caso único para medir o tempo de criação do prompt ---")

single_prompt_text = (
    'The argument "Exposure to violent video games causes at least a temporary increase in aggression and this exposure correlates with aggression in the real world" SUPPORT or ATTACK the belief "This house believes that the sale of violent video games to minors should be banned"?'
)

async def main():
    start_time = time.time()
    prompt_gerado = await create_prompt_with_paths_online(single_prompt_text)
    end_time = time.time()

    print(f"\n⏱️ Tempo de execução para gerar o prompt: {end_time - start_time:.4f} segundos")
    print("\n--- Prompt Gerado ---")
    print(prompt_gerado)

# Chamada principal para executar a função assíncrona
# Em um notebook Colab, basta usar 'await' na chamada da função async
await main()

In [ ]:
from conceptnet_lite import Label

In [ ]:
from peewee import OperationalError

In [ ]:
# ==============================================================================
# CÉLULA 7: TESTE DE CASO ÚNICO
# ==============================================================================
print("--- Executando teste de caso único para medir o tempo de criação do prompt ---")

import time

print("--- Executando teste de caso único para medir o tempo de criação do prompt ---")

# Corrigido o uso de aspas
'''
single_prompt_text = (
    'The argument "If, for any reason, the single child is unable to care for their older adult relatives, '
    'the oldest generations would face a lack of resources and necessities" SUPPORT or ATTACK the belief '
    '"This house supports the one-child policy of the republic of China"?'
)
'''

single_prompt_text = (
    'The argument "Exposure to violent video games causes at least a temporary increase in aggression and this exposure correlates with aggression in the real world" SUPPORT or ATTACK the belief "This house believes that the sale of violent video games to minors should be banned"?'
)

start_time = time.time()
prompt_gerado = create_prompt_with_paths_local(single_prompt_text)
end_time = time.time()

print(f"\n⏱️ Tempo de execução para gerar o prompt: {end_time - start_time:.4f} segundos")
print("\n--- Prompt Gerado ---")
print(prompt_gerado)

In [ ]:
# ==============================================================================
# CÉLULA 7: TESTE DE CASO ÚNICO ONLINE
# ==============================================================================
print("--- Executando teste de caso único para medir o tempo de criação do prompt ---")

import time

print("--- Executando teste de caso único para medir o tempo de criação do prompt ---")

# Corrigido o uso de aspas
'''
single_prompt_text = (
    'The argument "If, for any reason, the single child is unable to care for their older adult relatives, '
    'the oldest generations would face a lack of resources and necessities" SUPPORT or ATTACK the belief '
    '"This house supports the one-child policy of the republic of China"?'
)
'''

single_prompt_text = (
    'The argument "Exposure to violent video games causes at least a temporary increase in aggression and this exposure correlates with aggression in the real world" SUPPORT or ATTACK the belief "This house believes that the sale of violent video games to minors should be banned"?'
)

start_time = time.time()
prompt_gerado = create_prompt_with_paths_online(single_prompt_text)
end_time = time.time()

print(f"\n⏱️ Tempo de execução para gerar o prompt: {end_time - start_time:.4f} segundos")
print("\n--- Prompt Gerado ---")
print(prompt_gerado)

In [ ]:
# ==============================================================================
# CÉLULA 8: PROCESSAMENTO EM LOTE COM CHECKPOINTING
# ==============================================================================
prompts_filename = '/content/drive/My Drive/.../prompts_IBM.csv'
processed_prompts = set()

# Verifica se o arquivo já existe e carrega os prompts já processados
if os.path.exists(prompts_filename):
    print(f"Arquivo '{prompts_filename}' encontrado. Carregando prompts já processados...")
    df_existente = pd.read_csv(prompts_filename)
    processed_prompts = set(df_existente['prompt_question'])
    print(f"{len(processed_prompts)} prompts já haviam sido processados.")
else:
    print(f"Arquivo '{prompts_filename}' não encontrado. Um novo será criado.")

# Filtra o dataframe para processar apenas os prompts que ainda não estão no arquivo
prompts_a_processar = test_df[~test_df['question'].isin(processed_prompts)]

if prompts_a_processar.empty:
    print("Todos os prompts já foram processados. Nenhuma ação necessária.")
else:
    print(f"Iniciando o processamento de {len(prompts_a_processar)} novos prompts...")

    # Itera sobre os prompts restantes
    for index, row in prompts_a_processar.iterrows():
        start_time_prompt = time.time()

        question = row['question']
        right_answer = row['claims.stance']

        # Cria o prompt enriquecido
        full_prompt = create_prompt_with_paths_local(question)

        # Prepara os dados para salvar
        novo_prompt_df = pd.DataFrame([{
            'prompt_question': question, # A pergunta original
            'full_prompt_text': full_prompt, # O prompt completo com contexto
            'right_answer': right_answer
        }])

        # Anexa ao arquivo CSV
        # O cabeçalho só é escrito se o arquivo não existir
        header = not os.path.exists(prompts_filename)
        novo_prompt_df.to_csv(prompts_filename, mode='a', header=header, index=False)

        end_time_prompt = time.time()
        print(f"Prompt {index} processado e salvo em {end_time_prompt - start_time_prompt:.2f}s.")

print("Processamento em lote concluído.")

In [ ]:
import sys
import subprocess
import os

# --- Bloco de instalação de bibliotecas ---
print("Verificando e instalando bibliotecas necessárias...")
required_packages = ["aiohttp", "pandas", "nltk", "nest_asyncio"]
for package in required_packages:
    try:
        __import__(package)
        print(f"Biblioteca '{package}' já instalada.")
    except ImportError:
        print(f"Biblioteca '{package}' não encontrada. Instalando...")
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", package])
            print(f"Biblioteca '{package}' instalada com sucesso.")
        except subprocess.CalledProcessError as e:
            print(f"Erro ao instalar a biblioteca '{package}': {e}")
            sys.exit(1)

# Recarrega as bibliotecas após a instalação, se necessário
import aiohttp
import pandas as pd
import nltk
import nest_asyncio

# Aplica o patch para permitir loops de eventos aninhados, resolvendo o RuntimeError
nest_asyncio.apply()

# Baixa a lista de stopwords do NLTK, se ainda não estiver instalada
try:
    nltk.data.find('corpora/stopwords')
    print("NLTK stopwords já baixadas.")
except nltk.downloader.DownloadError:
    print("Baixando NLTK stopwords...")
    nltk.download('stopwords')

import asyncio
from requests.exceptions import RequestException, JSONDecodeError
from nltk.corpus import stopwords
import time
import random

# Placeholder para o seu DataFrame, caso ele não tenha sido carregado
try:
    # Tenta usar o DataFrame do ambiente do usuário
    test_df
except NameError:
    print("\nAviso: O DataFrame 'test_df' não foi encontrado. Criando um DataFrame vazio para demonstração.")
    # Exemplo de como você poderia carregar seu DataFrame, se necessário
    # test_df = pd.read_csv('caminho_para_o_seu_arquivo_ibm.csv')
    test_df = pd.DataFrame(columns=['question', 'claims.stance'])
else:
    print("DataFrame 'test_df' encontrado no ambiente.")


# ==============================================================================
# CÉLULA 6: FUNÇÕES ADAPTADAS PARA CONCEPTNET ONLINE (ASSÍNCRONAS COM LIMITADOR)
# ==============================================================================
# Relações a serem excluídas
excluded_relations = {'AtLocation', 'EtymologicallyRelated'}

def extract_key_terms(text):
    """
    Extrai termos-chave, incluindo frases de substantivos (chunks), para capturar conceitos mais complexos,
    utilizando a lista de stopwords do NLTK para filtragem automática.
    """
    tokens = nltk.word_tokenize(text.lower())

    english_stopwords = set(stopwords.words('english'))

    tagged_words = nltk.pos_tag(tokens, tagset='universal')
    key_terms = []

    single_words = [word for word, tag in tagged_words if tag in ('NOUN', 'VERB', 'ADJ') and word not in english_stopwords]
    key_terms.extend(list(set(single_words)))

    bigrams = nltk.bigrams(tokens)
    for bigram in bigrams:
        if bigram[0] not in english_stopwords and bigram[1] not in english_stopwords:
            if nltk.pos_tag([bigram[0]], tagset='universal')[0][1] in ('ADJ', 'NOUN') and \
               nltk.pos_tag([bigram[1]], tagset='universal')[0][1] in ('NOUN'):
                key_terms.append(' '.join(bigram))

    return list(set(key_terms))

def extract_entities_from_argument_and_topic(question_prompt):
    try:
        parts = question_prompt.split('SUPPORT or ATTACK the belief')
        argument_text = parts[0].replace('The argument', '').strip().strip('"')
        topic_text = parts[1].strip().strip('"?')

        argument_entities = extract_key_terms(argument_text)
        topic_entities = extract_key_terms(topic_text)

        return argument_entities, topic_entities
    except Exception as e:
        print(f"Erro ao parsear o prompt: {e}")
        return [], []

async def fetch_paths(session, semaphore, rate_limiter, arg_entity, topic_entity):
    """
    Função assíncrona para buscar caminhos entre duas entidades, usando semáforo e limitador de taxa.
    """
    if arg_entity == topic_entity:
        return []

    async with semaphore:
        await rate_limiter.wait()
        url = f"http://api.conceptnet.io/query?start=/c/en/{arg_entity.replace(' ', '_')}&end=/c/en/{topic_entity.replace(' ', '_')}"

        try:
            async with session.get(url) as response:
                response.raise_for_status()
                data = await response.json()

                paths = []
                for edge in data.get('edges', []):
                    start_label = edge['start']['label']
                    end_label = edge['end']['label']
                    rel_label = edge['rel']['label']

                    if rel_label not in excluded_relations and start_label != end_label:
                        paths.append(f"'{start_label}' -> ({rel_label}) -> '{end_label}'")
                return paths

        except (aiohttp.ClientError, asyncio.TimeoutError, JSONDecodeError) as e:
            print(f"Erro na requisição para a API do ConceptNet entre '{arg_entity}' e '{topic_entity}': {e}")
            return []

async def find_paths_between_entities_online_async(argument_entities, topic_entities):
    """
    Busca por caminhos entre entidades usando a API online do ConceptNet
    de forma assíncrona para maior velocidade.
    """
    paths_list = []

    all_entities = list(set(argument_entities + topic_entities))

    semaphore = asyncio.Semaphore(5)
    rate_limiter = RateLimiter(max_requests=120, period=60)

    tasks = [asyncio.ensure_future(fetch_paths(session, semaphore, rate_limiter, arg_entity, topic_entity))
             for arg_entity in all_entities for topic_entity in all_entities]

    results = await asyncio.gather(*tasks)

    for result_paths in results:
        paths_list.extend(result_paths)

    if paths_list:
        return ' '.join(sorted(list(set(paths_list))))
    else:
        return 'Nenhum caminho direto encontrado entre os conceitos do argumento e do tópico.'

async def create_prompt_with_paths_online(question_prompt):
    """
    Função assíncrona para criar o prompt completo.
    """
    argument_entities, topic_entities = extract_entities_from_argument_and_topic(question_prompt)

    argument_entities = list(set(argument_entities) - set(topic_entities))

    conceptnet_context = await find_paths_between_entities_online_async(argument_entities, topic_entities)

    system_instruction = ("Your task is to analyze an argument and a belief, using the provided 'Knowledge Graph Context' "
                          "which shows direct paths between concepts from the argument and the belief. "
                          "Based on this, decide if the argument attacks or supports the belief. "
                          "Your answer must be ONLY the word 'ATTACK' or 'SUPPORT'.")

    full_prompt = (f"{system_instruction}\\n\\n"
                   f"Knowledge Graph Context: {conceptnet_context}\\n\\n"
                   f"Question: {question_prompt}")

    return full_prompt

# ==============================================================================
# CÉLULA 8: PROCESSAMENTO EM LOTE COM CHECKPOINTING (ASSÍNCRONO)
# ==============================================================================
prompts_filename = '/content/drive/My Drive/.../prompts_IBM.csv'

async def batch_process_main(prompts_to_process, filename):
    total_requests_made = 0
    prompts_processed_count = 0
    start_time_batch = time.perf_counter()

    if prompts_to_process.empty:
        print("Todos os prompts já foram processados. Nenhuma ação necessária.")
        return

    print(f"\nIniciando o processamento de {len(prompts_to_process)} novos prompts de forma assíncrona...")

    async with aiohttp.ClientSession() as session:
        header = not os.path.exists(filename)

        for index, row in prompts_to_process.iterrows():
            question = row['question']
            right_answer = row['claims.stance']

            try:
                full_prompt, requests_count = await create_prompt_with_paths_online(question, session)

                novo_prompt_df = pd.DataFrame([{
                    'prompt_question': question,
                    'full_prompt_text': full_prompt,
                    'right_answer': right_answer
                }])

                novo_prompt_df.to_csv(filename, mode='a', header=header if prompts_processed_count == 0 else False, index=False)

                prompts_processed_count += 1
                total_requests_made += requests_count

                current_time = time.perf_counter()
                elapsed_time = current_time - start_time_batch

                print(f"--- Progresso do Prompt {prompts_processed_count} ---")
                print(f"Tempo decorrido: {elapsed_time:.2f} segundos")
                print(f"Requisições à API do ConceptNet feitas para este prompt: {requests_count}")
                print(f"Total de prompts processados: {prompts_processed_count}/{len(prompts_to_process)}")
                print(f"Total de requisições acumuladas: {total_requests_made}")
                print("-" * 30)

            except Exception as e:
                print(f"Erro ao processar o prompt: {question}. Erro: {e}")
                continue

    end_time_batch = time.perf_counter()
    print(f"\nProcessamento em lote concluído em {end_time_batch - start_time_batch:.2f}s.")
    print(f"Total de prompts processados: {prompts_processed_count}")
    print(f"Total de requisições feitas: {total_requests_made}")


async def main_execution():
    if os.path.exists(prompts_filename):
        print(f"Arquivo '{prompts_filename}' encontrado. Carregando prompts já processados...")
        df_existente = pd.read_csv(prompts_filename)
        processed_prompts = set(df_existente['prompt_question'])
        print(f"{len(processed_prompts)} prompts já haviam sido processados.")
    else:
        print(f"Arquivo '{prompts_filename}' não encontrado. Um novo será criado.")
        processed_prompts = set()

    if 'test_df' not in locals() and 'test_df' not in globals():
        print("Erro: DataFrame 'test_df' não encontrado. Por favor, carregue o seu DataFrame de teste.")
        prompts_a_processar = pd.DataFrame()
    else:
        prompts_a_processar = test_df[~test_df['question'].isin(processed_prompts)]

    await batch_process_main(prompts_a_processar, prompts_filename)

def run_main():
    try:
        asyncio.run(main_execution())
    except RuntimeError as e:
        if "Event loop is already running" in str(e):
            print("Event loop já está em execução. Usando nest_asyncio...")
            loop = asyncio.get_event_loop()
            if loop.is_running():
                loop.run_until_complete(main_execution())
            else:
                loop.run_until_complete(main_execution())
        else:
            raise e

# --- Bloco de execução principal ---
if __name__ == "__main__":
    run_main()

In [ ]:
# ==============================================================================
# CÉLULA 6: FUNÇÕES ADAPTADAS PARA O GOOGLE KNOWLEDGE GRAPH (ASSÍNCRONAS)
# ==============================================================================
import nltk
import requests
import asyncio
import aiohttp
import json
from requests.exceptions import RequestException, JSONDecodeError
from nltk.corpus import stopwords
import time

# Baixa a lista de stopwords do NLTK, se ainda não estiver instalada
try:
    nltk.data.find('corpora/stopwords')
except nltk.downloader.DownloadError:
    nltk.download('stopwords')

# --- Insira sua chave de API do Google aqui ---
# Exemplo: api_key = ''
api_key = ''

def extract_key_terms(text):
    """
    Extrai termos-chave, incluindo frases de substantivos (chunks), para capturar conceitos mais complexos,
    utilizando a lista de stopwords do NLTK para filtragem automática.
    """
    tokens = nltk.word_tokenize(text.lower())

    english_stopwords = set(stopwords.words('english'))

    tagged_words = nltk.pos_tag(tokens, tagset='universal')
    key_terms = []

    single_words = [word for word, tag in tagged_words if tag in ('NOUN', 'VERB', 'ADJ') and word not in english_stopwords]
    key_terms.extend(list(set(single_words)))

    bigrams = nltk.bigrams(tokens)
    for bigram in bigrams:
        if bigram[0] not in english_stopwords and bigram[1] not in english_stopwords:
            if nltk.pos_tag([bigram[0]], tagset='universal')[0][1] in ('ADJ', 'NOUN') and \
               nltk.pos_tag([bigram[1]], tagset='universal')[0][1] in ('NOUN'):
                key_terms.append(' '.join(bigram))

    return list(set(key_terms))

def extract_entities_from_argument_and_topic(question_prompt):
    try:
        parts = question_prompt.split('SUPPORT or ATTACK the belief')
        argument_text = parts[0].replace('The argument', '').strip().strip('"')
        topic_text = parts[1].strip().strip('"?')

        argument_entities = extract_key_terms(argument_text)
        topic_entities = extract_key_terms(topic_text)

        return argument_entities, topic_entities
    except Exception as e:
        print(f"Erro ao parsear o prompt: {e}")
        return [], []

async def fetch_entity_info(session, entity):
    """
    Função assíncrona para buscar informações de uma entidade na API do Google Knowledge Graph.
    """
    if not api_key:
        print("Erro: A chave da API não foi definida. Por favor, insira sua chave.")
        return f"Erro na API para o termo '{entity}'."

    # A API do Google é focada em busca, então usamos um termo por vez
    url = f"https://kgsearch.googleapis.com/v1/entities:search?query={entity.replace(' ', '%20')}&key={api_key}"

    try:
        async with session.get(url) as response:
            response.raise_for_status()
            data = await response.json()

            # Pega o primeiro resultado mais relevante
            if data and 'itemListElement' in data and data['itemListElement']:
                result = data['itemListElement'][0]['result']
                name = result.get('name', 'N/A')
                description = result.get('description', 'N/A')

                return f"Conceito '{name}' (tipo: {', '.join(result.get('@type', []))}) - Descrição: {description}."
            else:
                return f"Nenhuma informação encontrada para o termo '{entity}'."

    except (aiohttp.ClientError, asyncio.TimeoutError, JSONDecodeError) as e:
        print(f"Erro na requisição para a API do Google Knowledge Graph para o termo '{entity}': {e}")
        return f"Erro na API para o termo '{entity}'."

async def find_paths_between_entities_online_async(argument_entities, topic_entities):
    """
    Busca por informações das entidades usando a API do Google de forma assíncrona.
    """
    all_entities = list(set(argument_entities + topic_entities))

    async with aiohttp.ClientSession() as session:
        # Cria uma lista de tarefas para buscar cada entidade
        tasks = [asyncio.ensure_future(fetch_entity_info(session, entity))
                 for entity in all_entities]

        results = await asyncio.gather(*tasks)

    # Combina os resultados em uma única string de contexto
    if any("Nenhuma informação encontrada" not in r and "Erro na API" not in r for r in results):
        return ' '.join(list(set(results)))
    else:
        return 'Nenhum contexto relevante encontrado para os conceitos do argumento e do tópico.'

async def create_prompt_with_paths_online(question_prompt):
    """
    Função assíncrona para criar o prompt completo.
    """
    argument_entities, topic_entities = extract_entities_from_argument_and_topic(question_prompt)

    # Busca informações para todas as entidades
    conceptnet_context = await find_paths_between_entities_online_async(argument_entities, topic_entities)

    system_instruction = ("Your task is to analyze an argument and a belief, using the provided 'Knowledge Graph Context' "
                          "which shows direct paths between concepts from the argument and the belief. "
                          "Based on this, decide if the argument attacks or supports the belief. "
                          "Your answer must be ONLY the word 'ATTACK' or 'SUPPORT'.")

    full_prompt = (f"{system_instruction}\\n\\n"
                   f"Knowledge Graph Context: {conceptnet_context}\\n\\n"
                   f"Question: {question_prompt}")

    return full_prompt

# ==============================================================================
# CÉLULA 7: TESTE DE CASO ÚNICO ONLINE
# ==============================================================================
print("--- Executando teste de caso único para medir o tempo de criação do prompt ---")

single_prompt_text = (
    'The argument "Exposure to violent video games causes at least a temporary increase in aggression and this exposure correlates with aggression in the real world" SUPPORT or ATTACK the belief "This house believes that the sale of violent video games to minors should be banned"?'
)

async def main():
    start_time = time.time()
    prompt_gerado = await create_prompt_with_paths_online(single_prompt_text)
    end_time = time.time()

    print(f"\n⏱️ Tempo de execução para gerar o prompt: {end_time - start_time:.4f} segundos")
    print("\n--- Prompt Gerado ---")
    print(prompt_gerado)

# Chamada principal para executar a função assíncrona
# Em um notebook Colab, basta usar 'await' na chamada da função async
await main()

In [ ]:
from conceptnet_lite import Label

In [ ]:
from peewee import OperationalError

In [ ]:
# ==============================================================================
# CÉLULA 7: TESTE DE CASO ÚNICO
# ==============================================================================
print("--- Executando teste de caso único para medir o tempo de criação do prompt ---")

import time

print("--- Executando teste de caso único para medir o tempo de criação do prompt ---")

# Corrigido o uso de aspas
'''
single_prompt_text = (
    'The argument "If, for any reason, the single child is unable to care for their older adult relatives, '
    'the oldest generations would face a lack of resources and necessities" SUPPORT or ATTACK the belief '
    '"This house supports the one-child policy of the republic of China"?'
)
'''

single_prompt_text = (
    'The argument "Exposure to violent video games causes at least a temporary increase in aggression and this exposure correlates with aggression in the real world" SUPPORT or ATTACK the belief "This house believes that the sale of violent video games to minors should be banned"?'
)

start_time = time.time()
prompt_gerado = create_prompt_with_paths_local(single_prompt_text)
end_time = time.time()

print(f"\n⏱️ Tempo de execução para gerar o prompt: {end_time - start_time:.4f} segundos")
print("\n--- Prompt Gerado ---")
print(prompt_gerado)

In [ ]:
# ==============================================================================
# CÉLULA 7: TESTE DE CASO ÚNICO ONLINE
# ==============================================================================
print("--- Executando teste de caso único para medir o tempo de criação do prompt ---")

import time

print("--- Executando teste de caso único para medir o tempo de criação do prompt ---")

# Corrigido o uso de aspas
'''
single_prompt_text = (
    'The argument "If, for any reason, the single child is unable to care for their older adult relatives, '
    'the oldest generations would face a lack of resources and necessities" SUPPORT or ATTACK the belief '
    '"This house supports the one-child policy of the republic of China"?'
)
'''

single_prompt_text = (
    'The argument "Exposure to